# Hybrid Two-Tower TT0 / TT1 / TT2 — Kaggle

Notebook tự chứa, không cần upload `.py` hoặc clone repo. Chọn kernel mới Python >=3.10,
GPU T4 × 2 và Internet để cài dependencies. Giữ Torch/CUDA của image Kaggle.
Chạy một category trước để đo tài nguyên; mặc định All_Beauty và đủ ba variants.

| Input private | Dùng cho |
|---|---|
| thoandanh/gold-training-v2 | Tất cả variants |
| thoandanh/gold-emb-all-beauty-v2 | TT1/TT2 All_Beauty |
| thoandanh/gold-emb-video-games-v2 | TT1/TT2 Video_Games |
| thoandanh/gold-emb-industrial-and-scientific-v2 | TT1/TT2 Industrial_and_Scientific |

Thêm Dataset đúng ngành vào Input, pin version trong Kaggle và ghi INPUT_VERSIONS bên dưới.
Runtime verify complete, checksums, source export hash, encoder/revision và mappings riêng fold.
Không encode/fine-tune MiniLM. Đây là mở rộng theo Two-Tower retrieval của Google,
không phải tái lập loss Mixed Negative Sampling của paper.


In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ENVIRONMENT_READY = False
for variable in ['OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'OMP_NUM_THREADS']:
    os.environ[variable] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
if sys.version_info < (3, 10):
    raise RuntimeError('Cần Python >=3.10')
packages = (['numpy>=2.1,<3', 'scipy>=1.14.1,<2', 'pandas>=2.2.3,<4', 'pyarrow>=20,<24']
    if sys.version_info >= (3, 13) else ['numpy==1.26.4', 'scipy==1.14.1', 'pandas==2.2.3', 'pyarrow==20.0.0'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=:all:',
    *packages, 'psutil', 'matplotlib', 'kagglehub>=1.0.3,<2'], check=True)
subprocess.run([sys.executable, '-c', 'import torch,numpy,pandas,scipy,pyarrow,psutil; print(torch.__version__,torch.version.cuda,torch.cuda.device_count())'], check=True)
ENVIRONMENT_READY = True


## Hai encoder độc lập

TT0: User ID → MLP → vector; Item ID → MLP → vector.
TT1 thêm item text MiniLM 384 → Linear 16 và has_text vào item input.
TT2 thêm review profile 384 → Linear 16 và 7 thống kê rating/review vào user input.
TT2: user concat 31 → Linear 64 → ReLU → Linear 32;
item concat 25 → Linear 64 → ReLU → Linear 32. ID dim=8.
Output không L2 normalize, score = dot product, không thêm MLP tương tác sau hai tower.
Thiếu item text dùng zero + flag; item vẫn có ID và vẫn được xếp hạng.

Review profile là mean event-weighted rồi L2 normalize. Mỗi anchor chỉ lấy events
**strictly trước prediction_time**, loại cả group cùng timestamp; không dùng target review/rating.
Scaler fit positive train anchors riêng fold. Giữ mọi positive kể cả history rỗng.
Metadata sản phẩm là snapshot tĩnh, chưa chứng minh đúng phiên bản tại thời điểm lịch sử.


In [ ]:
%%writefile /kaggle/working/gold_recommendation.py
"""Shared, self-contained Gold loader, exact ranker and ALS/content experiments."""
import argparse
import hashlib
import importlib.metadata
import json
import math
import os
import re
import shutil
import tempfile
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import sparse

PROTOCOL = 'fixed_origin_first_next_all_reviews_6m_v1'
CATEGORIES = ('All_Beauty', 'Video_Games', 'Industrial_and_Scientific')
BOUNDS = {'validation': ('2021-01-01', '2021-07-01'), 'test': ('2022-01-01', '2022-07-01')}
KS = (10, 20)


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(block)
    return digest.hexdigest()


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def array_hash(array):
    array = np.asarray(array)
    digest = hashlib.sha256(str((array.shape, array.dtype.str)).encode())
    for start in range(0, len(array), 8192):
        digest.update(np.ascontiguousarray(array[start:start+8192]).tobytes())
    return digest.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+chr(10), encoding='utf-8')
    temporary.replace(path)


def child(root, relative):
    root = Path(root).resolve()
    path = (root/relative).resolve()
    if path == root or not path.is_relative_to(root):
        raise ValueError('Path escapes artifact root')
    return path


def disk_guard(directory, additional=0, reserve_gib=5):
    if not math.isfinite(reserve_gib) or reserve_gib < 0:
        raise ValueError('Invalid disk reserve')
    if shutil.disk_usage(directory).free < additional + reserve_gib * 1024**3:
        raise RuntimeError('Insufficient output disk; save Output and use another session')


def save_array(path, array):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix+'.tmp')
    with temporary.open('wb') as stream:
        np.save(stream, array, allow_pickle=False)
    temporary.replace(path)
    return sha256(path)


def versions():
    result = {}
    for name in ('numpy','scipy','pandas','pyarrow','torch','implicit'):
        try: result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: result[name] = None
    return result


class Monitor:
    """Sample process RSS and device-wide used VRAM, including non-Torch ALS allocations."""
    def __init__(self, devices=()):
        self.devices = devices; self.peak_rss = 0; self.vram = {}; self.stop = threading.Event()

    def sample(self):
        import psutil
        self.peak_rss = max(self.peak_rss, psutil.Process().memory_info().rss)
        if any(str(d).startswith('cuda:') for d in self.devices):
            import torch
            for device in self.devices:
                free, total = torch.cuda.mem_get_info(device)
                self.vram[device] = max(self.vram.get(device, 0), int(total-free))

    def __enter__(self):
        self.started = time.monotonic(); self.sample()
        def poll():
            while not self.stop.wait(1):
                self.sample()
        self.thread = threading.Thread(target=poll, daemon=True); self.thread.start()
        return self

    def __exit__(self, *_):
        self.stop.set(); self.thread.join(); self.sample()
        self.result = dict(seconds=round(time.monotonic()-self.started, 3),
            sampled_peak_rss_bytes=self.peak_rss, sampled_device_used_vram_bytes=self.vram,
            measurement='1s samples; device-wide VRAM includes all allocators, not exclusive process accounting')


class Gold:
    def __init__(self, root, category):
        self.root = Path(root).resolve(); self.category = category; self.checked = set()
        if category not in CATEGORIES: raise ValueError('Unknown category')
        self.manifest = read_json(self.root/'dataset_manifest.json')
        if self.manifest.get('status') != 'complete' or set(self.manifest['categories']) != set(CATEGORIES):
            raise ValueError('Gold Dataset must contain all three complete categories')
        for name, key in (('training_contract.json','contract_sha256'),('export_plan.json','plan_sha256'),('README.md','readme_sha256')):
            if sha256(self.root/name) != self.manifest[key]: raise ValueError('Gold metadata checksum mismatch')
        self.contract = read_json(self.root/'training_contract.json')
        protocol = self.contract['protocol']
        if protocol.get('status') != 'locked' or protocol['protocol_id'] != PROTOCOL or protocol['timezone'] != 'UTC':
            raise ValueError('Wrong Gold protocol/timezone')
        for fold, (start, end) in BOUNDS.items():
            if (protocol[fold+'_start'], protocol[fold+'_end']) != (start, end):
                raise ValueError('Do not change locked temporal split')
        self.directory = self.root/category
        if sha256(self.directory/'category_manifest.json') != self.manifest['categories'][category]:
            raise ValueError('Category manifest checksum mismatch')
        self.info = read_json(self.directory/'category_manifest.json')
        if self.info.get('status') != 'complete' or self.info['category'] != category:
            raise ValueError('Wrong/incomplete category')
        if sha256(self.directory/'text_export_manifest.json') != self.info['sources']['text_export_sha256']:
            raise ValueError('Text export manifest mismatch')
        self.signature = dict(dataset_manifest=sha256(self.root/'dataset_manifest.json'),
            category_manifest=sha256(self.directory/'category_manifest.json'), category=category)

    def files(self, relative):
        spec = self.info['tables'][relative]
        paths = []
        for entry in spec['files']:
            path = child(self.directory, entry['path'])
            if path.parent != child(self.directory, relative) or path.suffix != '.parquet':
                raise ValueError('Unexpected table shard')
            if path not in self.checked:
                if path.stat().st_size != entry['bytes'] or sha256(path) != entry['sha256']:
                    raise ValueError('Gold shard checksum mismatch: '+relative)
                if pq.ParquetFile(path).metadata.num_rows != entry['rows']:
                    raise ValueError('Gold shard row count mismatch')
                self.checked.add(path)
            paths.append(path)
        if not paths or len(set(paths)) != len(paths) or sum(e['rows'] for e in spec['files']) != spec['rows']:
            raise ValueError('Invalid Gold shard list')
        return paths

    def table(self, relative, columns):
        table = pq.read_table(self.files(relative), columns=columns)
        if table.num_rows != self.info['tables'][relative]['rows']: raise ValueError('Gold read count mismatch')
        return table.to_pandas()

    def fold(self, name):
        return Fold(self, name)


class Fold:
    def __init__(self, gold, name):
        if name not in BOUNDS: raise ValueError('Unknown fold')
        self.gold = gold; self.name = name; self.start, self.end = BOUNDS[name]
        load = lambda table, cols: gold.table(name+'/'+table, cols)
        self.items = load('items',['parent_asin','item_idx']).sort_values('item_idx').reset_index(drop=True)
        users = load('users',['user_idx'])
        self.nusers, self.nitems = len(users), len(self.items)
        if not np.array_equal(np.sort(users.user_idx), np.arange(self.nusers)) or not np.array_equal(self.items.item_idx, np.arange(self.nitems)) or self.items.parent_asin.duplicated().any():
            raise ValueError('Mappings must be unique contiguous train-only indices')
        del users
        candidates = load('candidates',['parent_asin','item_idx']).sort_values('item_idx').reset_index(drop=True)
        if not candidates.equals(self.items): raise ValueError('Candidates differ from training item mapping')
        pairs = load('train_pairs',['user_idx','item_idx','strength','first_pair_time'])
        if pairs.isna().any().any() or not (pairs.strength == 1).all() or (pairs.first_pair_time >= pd.Timestamp(self.start)).any():
            raise ValueError('Invalid binary training pairs/cutoff')
        rows = pairs.user_idx.to_numpy(np.int32); cols = pairs.item_idx.to_numpy(np.int32)
        if (rows < 0).any() or (rows >= self.nusers).any() or (cols < 0).any() or (cols >= self.nitems).any():
            raise ValueError('Training index out of range')
        self.binary = sparse.csr_matrix((np.ones(len(pairs),np.float32),(rows,cols)), shape=(self.nusers,self.nitems))
        self.binary.sum_duplicates(); self.binary.sort_indices()
        if self.binary.nnz != len(pairs) or not (self.binary.data == 1).all(): raise ValueError('Duplicate train pairs')
        del pairs, rows, cols
        self.queries = load('queries',['query_id','user_idx','prediction_time','history_items','unknown_user'])
        if self.queries.query_id.duplicated().any() or self.queries.query_id.isna().any() or not (self.queries.prediction_time == pd.Timestamp(self.start)).all():
            raise ValueError('Invalid query keys/cutoff')
        self.user_indices = self.queries.user_idx.fillna(-1).to_numpy(np.int32)
        if (self.user_indices < -1).any() or (self.user_indices >= self.nusers).any(): raise ValueError('Invalid query user index')
        if not np.array_equal(self.user_indices < 0, self.queries.unknown_user.to_numpy(bool)):
            raise ValueError('Cold-user flag mismatch')
        history_counts=np.zeros(len(self.queries),np.int64)
        known=self.user_indices>=0
        history_counts[known]=np.diff(self.binary.indptr)[self.user_indices[known]]
        if not np.array_equal(history_counts,self.queries.history_items.to_numpy()):
            raise ValueError('Query history support differs from training pairs')
        self.nqueries = len(self.queries)
        self.cohorts = load('cohorts',['query_id','warm_full','target_count','targets_in','repeat_targets','history_items'])
        if self.cohorts.query_id.duplicated().any(): raise ValueError('Duplicate cohort query')
        self.cohorts = self.queries[['query_id']].merge(self.cohorts,on='query_id',how='left',validate='one_to_one')
        if self.cohorts.isna().any().any() or not np.array_equal(self.cohorts.history_items,self.queries.history_items): raise ValueError('Missing/mismatched query cohort')
        labels = load('labels',['query_id','parent_asin','item_idx','target_time','in_candidates'])
        if labels[['query_id','parent_asin']].isna().any().any() or labels.duplicated(['query_id','parent_asin']).any():
            raise ValueError('Duplicate/null target key')
        if labels.target_time.isna().any() or labels.in_candidates.isna().any() or ((labels.target_time < pd.Timestamp(self.start)) | (labels.target_time >= pd.Timestamp(self.end))).any():
            raise ValueError('Target outside evaluation window')
        qmap = pd.Series(np.arange(self.nqueries),index=self.queries.query_id)
        imap = pd.Series(np.arange(self.nitems),index=self.items.parent_asin)
        qrows = labels.query_id.map(qmap)
        if qrows.isna().any(): raise ValueError('Label with no query')
        qrows = qrows.to_numpy(np.int32)
        indices = labels.parent_asin.map(imap).fillna(-1).to_numpy(np.int32)
        if not np.array_equal(indices >= 0, labels.in_candidates.to_numpy(bool)) or not np.array_equal(indices,labels.item_idx.fillna(-1).to_numpy(np.int32)):
            raise ValueError('Label candidate mapping mismatch')
        self.target_counts = np.bincount(qrows,minlength=self.nqueries)
        if (self.target_counts == 0).any() or not np.array_equal(self.target_counts,self.cohorts.target_count):
            raise ValueError('Empty/mismatched label population')
        inside = indices >= 0
        self.truth = sparse.csr_matrix((np.ones(inside.sum(),np.float32),(qrows[inside],indices[inside])), shape=(self.nqueries,self.nitems))
        if not np.array_equal(np.asarray(self.truth.sum(axis=1)).ravel(),self.cohorts.targets_in): raise ValueError('Target coverage mismatch')
        self.warm = self.cohorts.warm_full.to_numpy(bool)
        if not np.array_equal(self.warm,(self.user_indices >= 0) & (self.cohorts.targets_in.to_numpy()==self.target_counts)):
            raise ValueError('Warm-full definition changed')
        expected = gold.info['folds'][name]
        if (self.binary.nnz,self.nqueries,int(self.warm.sum())) != (expected['train_pairs'],expected['queries'],expected['warm_full_queries']):
            raise ValueError('Population differs from Gold manifest')
        counts = np.asarray(self.binary.sum(axis=0)).ravel()
        self.popularity = np.lexsort((np.arange(self.nitems),-counts)).astype(np.int32)
        print(f'{name}: users={self.nusers:,}; items={self.nitems:,}; pairs={self.binary.nnz:,}; queries={self.nqueries:,}; warm-full={self.warm.sum():,}',flush=True)

    def copy_mappings(self, root):
        for table in ('users','items'):
            for path in self.gold.files(self.name+'/'+table):
                target = Path(root)/'mappings'/table/path.name
                target.parent.mkdir(parents=True,exist_ok=True)
                if not target.exists(): shutil.copyfile(path,target)
                if sha256(target) != sha256(path): raise ValueError('Saved mapping corrupted')


def normalize_rows(matrix):
    matrix = np.asarray(matrix,dtype=np.float32)
    if not np.isfinite(matrix).all(): raise ValueError('Non-finite profile/vector')
    norms = np.linalg.norm(matrix,axis=1)
    valid = norms > 1e-12
    matrix[valid] /= norms[valid,None]
    matrix[~valid] = 0
    return matrix, valid


def content_profiles(binary, query_user_indices, item_vectors, item_valid):
    positions = np.flatnonzero(query_user_indices >= 0)
    histories = binary[query_user_indices[positions]]
    counts = np.asarray(histories @ item_valid.astype(np.float32)).ravel()
    profiles = np.asarray(histories @ item_vectors,dtype=np.float32)
    profiles /= np.maximum(counts,1)[:,None]
    profiles, valid = normalize_rows(profiles)
    valid &= counts > 0
    return positions[valid], profiles[valid], dict(known_queries=len(positions), usable_profile_queries=int(valid.sum()),
        queries_with_no_usable_history=int((counts==0).sum()), usable_history_items_sum=int(counts.sum()))


def load_item_vectors(gold, fold, embedding_root, bundle_name=None):
    export_hash = sha256(gold.directory/'text_export_manifest.json')
    matches = []
    for path in Path(embedding_root).rglob('embedding_manifest.json'):
        manifest = read_json(path)
        if manifest.get('signature',{}).get('export_manifest_sha256') == export_hash and (bundle_name is None or path.parent.name == bundle_name):
            matches.append((path,manifest))
    if len(matches) != 1: raise ValueError('Need exactly one matching embedding bundle; set EMBEDDING_ROOT/BUNDLE_NAME. Found: '+str([p.parent.name for p,_ in matches]))
    path, manifest = matches[0]; signature = manifest['signature']; encoder = gold.contract['encoder']
    if manifest.get('status') != 'complete': raise ValueError('Embedding bundle is partial; finish encoding first')
    for key in ('model','revision','dimension','max_seq_length','normalize_embeddings','output_dtype','long_text_policy'):
        if signature.get(key) != encoder[key]: raise ValueError('Embedding encoder mismatch: '+key)
    if signature.get('cleaning_version') != 'gold_text_nfc_html_whitespace_v1' or encoder['dimension'] != 384:
        raise ValueError('Wrong cleaning/dimension')
    refs = gold.table('item_text_refs',['parent_asin','text_id','has_text'])
    if refs.parent_asin.duplicated().any() or not np.array_equal(refs.text_id.notna(),refs.has_text): raise ValueError('Invalid item text refs')
    merged = fold.items.merge(refs,on='parent_asin',how='left',validate='one_to_one')
    if merged.has_text.isna().any(): raise ValueError('Candidate missing item ref')
    wanted = set(merged.loc[merged.has_text,'text_id'])
    found = {}; listed = manifest['shards']
    if len({s['path'] for s in listed}) != len(listed) or sum(s['rows'] for s in listed) != manifest['texts'] or len(listed) != manifest['total_shards']:
        raise ValueError('Incomplete/duplicate embedding shard list')
    for i,state in enumerate(listed,1):
        shard = child(path.parent,state['path'])
        if not state['path'].startswith('texts/') or shard.suffix != '.parquet' or state.get('signature') != signature or state.get('status') != 'complete':
            raise ValueError('Unexpected embedding shard')
        if read_json(shard.with_suffix('.json')) != {k:v for k,v in state.items() if k!='path'} or sha256(shard) != state['output_sha256']:
            raise ValueError('Embedding checkpoint/checksum mismatch')
        parquet = pq.ParquetFile(shard)
        if parquet.metadata.num_rows != state['rows']: raise ValueError('Embedding row count mismatch')
        for batch in parquet.iter_batches(batch_size=1024,columns=['text_id','embedding']):
            ids = batch.column(0).to_pylist()
            selected = [j for j,key in enumerate(ids) if key in wanted]
            if not selected: continue
            picked = batch.take(pa.array(selected,pa.int32()))
            column = picked.column(1)
            if not pa.types.is_fixed_size_list(column.type) or column.type.list_size != 384 or column.type.value_type != pa.float32() or column.null_count or column.flatten().null_count:
                raise ValueError('Wrong vector schema/null value')
            vectors = column.flatten().to_numpy(zero_copy_only=False).reshape(len(selected),384)
            if not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=0.002): raise ValueError('Invalid item vector norm')
            for key,vector in zip(picked.column(0).to_pylist(),vectors):
                if key in found: raise ValueError('Duplicate required text_id across embedding shards')
                found[key] = vector.copy()
        if i == 1 or i % 20 == 0 or i == len(listed): print(f'Embedding scan {i}/{len(listed)}; matched {len(found):,}/{len(wanted):,} item texts',flush=True)
    if set(found) != wanted: raise ValueError('has_text=True item vectors missing from complete embedding bundle')
    vectors = np.zeros((fold.nitems,384),np.float32)
    valid = merged.has_text.to_numpy(bool)
    for row in merged.loc[valid,['item_idx','text_id']].itertuples(index=False): vectors[row.item_idx] = found[row.text_id]
    return vectors, valid, dict(manifest_sha256=sha256(path), bundle=path.parent.name,
        candidate_items=fold.nitems, items_with_vectors=int(valid.sum()), distinct_item_texts=len(wanted), encoder=encoder)


def numpy_topk(scores, ids, k):
    scores = np.asarray(scores,dtype=np.float32)
    ids = np.broadcast_to(np.asarray(ids,dtype=np.int32),scores.shape)
    order = np.lexsort((ids,-scores),axis=1)[:,:min(k,scores.shape[1])]
    return np.take_along_axis(ids,order,1),np.take_along_axis(scores,order,1)


def torch_topk(scores, start, k):
    """Exact score top-K; resolve boundary ties by ascending item index."""
    import torch
    n = scores.shape[1]; take = min(k,n)
    values, indices = torch.topk(scores,min(k+1,n),dim=1,sorted=True)
    selected = indices[:,:take].clone()
    if n > take:
        ties = (values[:,take-1] == values[:,take]) & torch.isfinite(values[:,take-1])
        if ties.any(): selected[ties] = torch.argsort(scores[ties],dim=1,descending=True,stable=True)[:,:take]
    values = torch.gather(scores,1,selected)
    ids = selected.cpu().numpy().astype(np.int32)+start
    return numpy_topk(values.cpu().numpy(),ids,take)


def rank_exact(queries, items, valid_items, k=20, query_batch=512, item_block=16384, device='cpu'):
    if query_batch < 1 or item_block < 1 or not np.isfinite(queries).all() or not np.isfinite(items).all(): raise ValueError('Invalid ranker input')
    out_ids = np.full((len(queries),k),-1,np.int32); out_scores = np.full((len(queries),k),-np.inf,np.float32)
    use_gpu = str(device).startswith('cuda:')
    if use_gpu:
        import torch
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.set_float32_matmul_precision('highest')
        item_tensor = torch.as_tensor(items,device=device,dtype=torch.float32)
        mask_tensor = torch.as_tensor(valid_items,device=device,dtype=torch.bool)
    for start in range(0,len(queries),query_batch):
        end = min(len(queries),start+query_batch)
        best_ids = np.empty((end-start,0),np.int32); best_scores = np.empty((end-start,0),np.float32)
        if use_gpu: query_tensor = torch.as_tensor(queries[start:end],device=device,dtype=torch.float32)
        for lo in range(0,len(items),item_block):
            hi = min(len(items),lo+item_block)
            if use_gpu:
                with torch.inference_mode():
                    scores = query_tensor @ item_tensor[lo:hi].T
                    scores[:,~mask_tensor[lo:hi]] = -torch.inf
                    idx,val = torch_topk(scores,lo,k)
            else:
                scores = queries[start:end] @ items[lo:hi].T
                scores[:,~valid_items[lo:hi]] = -np.inf
                idx,val = numpy_topk(scores,np.arange(lo,hi),k)
            best_ids,best_scores = numpy_topk(np.concatenate((best_scores,val),axis=1),np.concatenate((best_ids,idx),axis=1),k)
        out_ids[start:end,:best_ids.shape[1]] = np.where(np.isfinite(best_scores),best_ids,-1)
        out_scores[start:end,:best_scores.shape[1]] = best_scores
    return out_ids,out_scores


def rank_checkpoints(query_positions, queries, items, item_valid, root, signature, config):
    root = Path(root); root.mkdir(parents=True,exist_ok=True)
    devices = config['score_devices']
    signature = dict(signature, query_positions=array_hash(query_positions), query_vectors=array_hash(queries),
        item_vectors=array_hash(items), item_valid=array_hash(item_valid), score_dtype='float32',
        backend='cuda' if any(d.startswith('cuda:') for d in devices) else 'numpy')
    digest = fingerprint(signature)
    shard_rows = config['score_shard_queries']
    output = np.full((len(query_positions),20),-1,np.int32)
    def work(task):
        index,start,end = task; path=root/f'part-{index:05d}.npz'; meta=path.with_suffix('.json')
        if path.exists() and meta.exists():
            state=read_json(meta)
            if state.get('signature') != digest or state.get('sha256') != sha256(path): raise ValueError('Scoring checkpoint mismatch')
            with np.load(path,allow_pickle=False) as data:
                if not np.array_equal(data['query_positions'],query_positions[start:end]) or data['items'].shape != (end-start,20): raise ValueError('Wrong checkpoint query range')
                predictions=data['items'].copy()
        else:
            device=devices[index%len(devices)]
            predictions,scores=rank_exact(queries[start:end],items,item_valid,query_batch=config['query_batch'],item_block=config['item_block'],device=device)
            disk_guard(root,reserve_gib=config['min_free_gib'])
            temporary=path.with_suffix('.npz.tmp')
            with temporary.open('wb') as stream: np.savez(stream,query_positions=query_positions[start:end],items=predictions,scores=scores)
            temporary.replace(path)
            save_json(meta,dict(signature=digest,sha256=sha256(path),rows=end-start))
        output[start:end] = predictions
        print(f'Scored/reused queries {end:,}/{len(queries):,}',flush=True)
    tasks=[(i,start,min(len(queries),start+shard_rows)) for i,start in enumerate(range(0,len(queries),shard_rows))]
    # One long-lived worker per device; no two jobs simultaneously allocate on one GPU.
    def device_work(rank):
        for task in tasks[rank::len(devices)]: work(task)
    with ThreadPoolExecutor(max_workers=len(devices)) as executor:
        list(executor.map(device_work,range(len(devices))))
    save_json(root/'manifest.json',dict(status='complete',signature=digest,queries=len(query_positions),shards=len(tasks),query_batch=config['query_batch'],item_block=config['item_block']))
    return output


def with_popularity(native, popularity):
    output = native.copy(); fallback = np.zeros(len(native),bool)
    limit = min(native.shape[1],len(popularity))
    for row in np.flatnonzero((native[:,:limit] < 0).any(axis=1)):
        chosen=list(native[row][native[row]>=0]); used=set(chosen)
        for item in popularity:
            if len(chosen) >= limit: break
            if int(item) not in used: chosen.append(int(item)); used.add(int(item))
        output[row,:len(chosen)] = chosen
        fallback[row] = True
    return output,fallback


def query_metrics(fold, predictions):
    predictions=np.asarray(predictions,dtype=np.int32)
    if predictions.shape != (fold.nqueries,20) or (predictions >= fold.nitems).any() or (predictions < -1).any(): raise ValueError('Invalid predictions')
    sorted_ids=np.sort(predictions,axis=1)
    if ((sorted_ids[:,1:] == sorted_ids[:,:-1]) & (sorted_ids[:,1:]>=0)).any(): raise ValueError('Duplicate recommended item')
    safe=np.maximum(predictions,0)
    matched=fold.truth[np.arange(fold.nqueries)[:,None],safe]
    hits=(matched.toarray() if sparse.issparse(matched) else np.asarray(matched)) > 0
    hits &= predictions >= 0
    result={}
    for k in KS:
        result[f'recall@{k}']=hits[:,:k].sum(axis=1)/fold.target_counts
        weights=1/np.log2(np.arange(k)+2)
        ideal=np.cumsum(weights)[np.minimum(k,fold.target_counts)-1]
        result[f'ndcg@{k}']=(hits[:,:k]*weights).sum(axis=1)/ideal
    result['hit@10']=hits[:,:10].any(axis=1).astype(float)
    return result


def group_masks(fold):
    history=fold.queries.history_items.to_numpy()
    masks={'all_queries':np.ones(fold.nqueries,bool),'warm_full':fold.warm}
    masks.update({'history_0':history==0,'history_1':history==1,'history_2':history==2,'history_3_4':(history>=3)&(history<=4),'history_5_plus':history>=5})
    repeats=fold.cohorts.repeat_targets.to_numpy()
    masks.update(target_novel=repeats==0,target_repeat=repeats==fold.target_counts,target_mixed=(repeats>0)&(repeats<fold.target_counts))
    return masks


def paired_ci(delta, samples=1000, seed=42):
    if len(delta)==0: return None
    rng=np.random.default_rng(seed)
    means=np.empty(samples)
    for i in range(samples): means[i]=np.mean(delta[rng.integers(0,len(delta),size=len(delta),dtype=np.int32)])
    lo,hi=np.quantile(means,[0.025,0.975])
    return dict(mean_delta=float(np.mean(delta)),ci95=[float(lo),float(hi)],bootstrap_samples=samples,queries=len(delta))


def evaluate(fold, native, root, model, config):
    root=Path(root); root.mkdir(parents=True,exist_ok=True)
    system,fallback=with_popularity(native,fold.popularity)
    popular=np.full(native.shape,-1,np.int32)
    popular[:,:min(20,fold.nitems)]=fold.popularity[:20]
    masks=group_masks(fold); rows=[]; per_query={}; native_available=(native>=0).any(axis=1)
    for name,predictions,used in (('popularity',popular,np.zeros(fold.nqueries,bool)),(model+'_native',native,np.zeros(fold.nqueries,bool)),(model+'_popularity',system,fallback)):
        measures=query_metrics(fold,predictions)
        reasons=np.full(fold.nqueries,'none',dtype=object)
        reasons[used & ~native_available & (fold.user_indices<0)]='cold_user'
        reasons[used & ~native_available & (fold.user_indices>=0)]='no_usable_content_profile' if model=='content' else 'zero_or_invalid_user_factor'
        reasons[used & native_available]='partial_list_fill'
        frame=pd.DataFrame(dict(query_id=fold.queries.query_id,warm_full=fold.warm,native_available=native_available,
            fallback_used=used,fallback_reason=reasons,**measures))
        frame.to_parquet(root/(name+'_per_query.parquet'),index=False)
        per_query[name]=measures
        for group,mask in masks.items():
            count=int(mask.sum())
            selected=predictions[mask,:10]; distinct=np.unique(selected[selected>=0])
            row=dict(model=name,cohort=group,queries=count,catalog_coverage_at10=len(distinct)/fold.nitems if count else None,
                prediction_coverage=float((predictions[mask]>=0).any(axis=1).mean()) if count else None,
                native_coverage=float(native_available[mask].mean()) if count and name!='popularity' else None,
                fallback_rate=float(used[mask].mean()) if count else None,
                candidate_target_coverage=float(fold.cohorts.targets_in.to_numpy()[mask].sum()/fold.target_counts[mask].sum()) if count else None)
            row.update({key:float(value[mask].mean()) if count else None for key,value in measures.items()})
            rows.append(row)
        # Compact indices are decoded with the saved fold item mapping, not row order from another fold.
        schema=pa.schema([('query_id',pa.string()),('item_indices',pa.list_(pa.int32())),('fallback',pa.bool_())])
        with pq.ParquetWriter(root/(name+'_top20.parquet'),schema,compression='snappy') as writer:
            for start in range(0,fold.nqueries,4096):
                end=min(fold.nqueries,start+4096)
                writer.write_table(pa.Table.from_arrays([pa.array(fold.queries.query_id.iloc[start:end]),
                    pa.array(predictions[start:end].tolist(),type=pa.list_(pa.int32())),pa.array(used[start:end])],schema=schema))
    pd.DataFrame(rows).to_csv(root/'metrics.csv',index=False)
    confidence={}
    for name in (model+'_native',model+'_popularity'):
        for group in ('all_queries','warm_full'):
            mask=masks[group]
            delta=per_query[name]['ndcg@10'][mask]-per_query['popularity']['ndcg@10'][mask]
            confidence[name+'/'+group]=paired_ci(delta,config['bootstrap_samples'],config['seed'])
    save_json(root/'confidence_vs_popularity.json',confidence)
    save_json(root/'evaluation_manifest.json',dict(status='complete',source=fold.gold.signature,fold=fold.name,
        queries=fold.nqueries,warm_full=int(fold.warm.sum()),model=model,metric_policy='macro query; targets outside candidates retained; repeats allowed',metrics=rows))
    return rows


def preflight(config):
    if config['algorithm'] not in ('als','content') or config['phase'] not in BOUNDS or config['category'] not in CATEGORIES:
        raise ValueError('Invalid algorithm/phase/category')
    for key in ('query_batch','item_block','score_shard_queries','bootstrap_samples','cpu_threads'):
        if type(config[key]) is not int or config[key] < 1: raise ValueError('Positive integer required: '+key)
    devices=config['score_devices']
    if not devices or len(set(devices)) != len(devices) or ('cpu' in devices and devices!=['cpu']):
        raise ValueError('Use unique CUDA devices, or [cpu] explicitly')
    hardware=dict(python=os.sys.version.split()[0],score_devices=devices,gpus=[])
    if any(d!='cpu' for d in devices):
        import torch
        cuda_available=torch.cuda.is_available()
        gpu_count=torch.cuda.device_count()
        print('CUDA diagnostics:',dict(torch=torch.__version__,torch_cuda=torch.version.cuda,
            cuda_available=cuda_available,gpu_count=gpu_count,requested=devices),flush=True)
        for device in devices:
            if not device.startswith('cuda:') or not device[5:].isdigit():
                raise ValueError('Invalid scoring device: '+str(device)+'; use cuda:0, cuda:1, ... or cpu')
            if not cuda_available or int(device[5:]) >= gpu_count:
                raise RuntimeError(f'Scoring requested {devices}, but CUDA available={cuda_available}, visible GPUs={gpu_count}, '
                    f'Torch={torch.__version__}, Torch CUDA={torch.version.cuda}. '
                    "Enable the Kaggle GPU accelerator and restart the session; with one usable GPU set SCORE_DEVICES=['cuda:0']; "
                    "for CPU scoring explicitly set SCORE_DEVICES=['cpu']. ALS_BACKEND does not control scoring.")
        torch.backends.cuda.matmul.allow_tf32=False
        torch.set_float32_matmul_precision('highest')
        hardware['gpus']=[dict(device=d,name=torch.cuda.get_device_name(d),total_memory=torch.cuda.get_device_properties(d).total_memory) for d in devices]
        hardware['torch_cuda']=torch.version.cuda
    if config['algorithm']=='als':
        try:
            import implicit
        except ModuleNotFoundError as error:
            if error.name != 'implicit': raise
            raise RuntimeError('Missing implicit in training interpreter. Run the dependency installation cell successfully before training.') from error
        if implicit.__version__ not in ('0.7.2','0.7.3'): raise RuntimeError('Supported implicit versions: 0.7.2 / 0.7.3')
        if config['als_backend']=='gpu':
            import implicit.gpu
            if not implicit.gpu.HAS_CUDA: raise RuntimeError('implicit CUDA backend unavailable. Version 0.7.3 requires CUDA 13 + RMM (install implicit[gpu]==0.7.3). Verify GPU/runtime support, or explicitly use ALS_BACKEND=cpu; GPU scoring remains independent. No silent fallback.')
            from implicit.gpu.als import AlternatingLeastSquares
            probe=AlternatingLeastSquares(factors=32,iterations=2,regularization=0.1,alpha=1,random_state=config['seed'])
            probe.fit(sparse.eye(3,format='csr',dtype=np.float32)*2,show_progress=False)
            cpu_probe=probe.to_cpu()
            if not np.isfinite(cpu_probe.user_factors).all(): raise RuntimeError('GPU ALS preflight produced invalid factors')
            del probe,cpu_probe
            hardware['als_fit_device']='cuda:0'
        elif config['als_backend']!='cpu': raise ValueError('ALS_BACKEND must be gpu or cpu')
    print('Runtime:',versions(),'scoring:',devices,'algorithm:',config['algorithm'],
        'ALS backend:',config.get('als_backend') if config['algorithm']=='als' else 'not used',flush=True)
    return hardware


def fit_als(fold, parameters, directory, signature, config):
    directory=Path(directory); directory.mkdir(parents=True,exist_ok=True)
    manifest_path=directory/'fit_manifest.json'
    expected=fingerprint(dict(source=signature,fold=fold.name,parameters=parameters,seed=config['seed'],backend=config['als_backend']))
    if manifest_path.exists():
        state=read_json(manifest_path)
        if state.get('signature')!=expected: raise ValueError('ALS model source/config mismatch')
        if state.get('status')=='complete' and not state.get('discarded'):
            for name in ('user_factors','item_factors'):
                if sha256(directory/(name+'.npy')) != state['hashes'][name]: raise ValueError('ALS factor checkpoint corrupted')
            return np.load(directory/'user_factors.npy',allow_pickle=False),np.load(directory/'item_factors.npy',allow_pickle=False),state
    confidence=fold.binary.copy()
    confidence.data.fill(1+parameters['confidence_alpha'])
    if config['als_backend']=='gpu':
        from implicit.gpu.als import AlternatingLeastSquares
        model=AlternatingLeastSquares(factors=parameters['factors'],regularization=parameters['regularization'],
            alpha=1.0,iterations=parameters['iterations'],random_state=config['seed'],dtype=np.float32,calculate_training_loss=True)
    else:
        from implicit.cpu.als import AlternatingLeastSquares
        model=AlternatingLeastSquares(factors=parameters['factors'],regularization=parameters['regularization'],
            alpha=1.0,iterations=parameters['iterations'],random_state=config['seed'],dtype=np.float32,
            num_threads=config['cpu_threads'],calculate_training_loss=True)
    save_json(manifest_path,dict(status='running',signature=expected,parameters=parameters))
    def progress(epoch,elapsed,loss):
        save_json(directory/'fit_progress.json',dict(completed_iterations=int(epoch)+1,iteration_seconds=float(elapsed),
            loss=float(loss) if loss is not None and np.isfinite(loss) else None))
    devices=config['score_devices'] if config['als_backend']=='gpu' else []
    with Monitor(devices) as monitor:
        model.fit(confidence,show_progress=True,callback=progress)
        if config['als_backend']=='gpu': model=model.to_cpu()
        users=np.asarray(model.user_factors,dtype=np.float32); items=np.asarray(model.item_factors,dtype=np.float32)
    if users.shape!=(fold.nusers,parameters['factors']) or items.shape!=(fold.nitems,parameters['factors']) or not np.isfinite(users).all() or not np.isfinite(items).all():
        raise ValueError('Invalid ALS factor shapes/values')
    disk_guard(directory,users.nbytes+items.nbytes,config['min_free_gib'])
    hashes={name:save_array(directory/(name+'.npy'),array) for name,array in (('user_factors',users),('item_factors',items))}
    state=dict(status='complete',signature=expected,parameters=parameters,seed=config['seed'],backend=config['als_backend'],
        fold=fold.name,train_pairs=int(fold.binary.nnz),confidence_policy='observed confidence=1+confidence_alpha; absent=1; library alpha=1',hashes=hashes,resources=monitor.result)
    save_json(manifest_path,state)
    return users,items,state


def als_predictions(fold, users, items, root, signature, config, only_warm=False):
    eligible=fold.user_indices>=0
    if only_warm: eligible &= fold.warm
    positions=np.flatnonzero(eligible)
    vectors=users[fold.user_indices[positions]]
    valid=np.isfinite(vectors).all(axis=1) & (np.linalg.norm(vectors,axis=1)>1e-12)
    positions=positions[valid]; vectors=vectors[valid]
    predictions=np.full((fold.nqueries,20),-1,np.int32)
    predictions[positions]=rank_checkpoints(positions,vectors,items,np.ones(fold.nitems,bool),root,signature,config)
    return predictions


def selection_score(result):
    p=result['parameters']
    return (result['ndcg@10'],result['recall@10'],-p['factors'])


def warm_score(fold,predictions):
    if not fold.warm.any(): raise ValueError('No warm-full validation queries')
    metrics=query_metrics(fold,predictions)
    return {key:float(value[fold.warm].mean()) for key,value in metrics.items()}


def tune_als(fold, root, signature, config):
    root=Path(root); completed=[]
    for index in range(5):
        if index==0: params=dict(factors=16,regularization=0.1,confidence_alpha=1,iterations=5)
        elif index in (1,2): params=dict(factors=64,regularization=0.1,confidence_alpha=10 if index==1 else 40,iterations=20)
        elif index==3:
            preferred=max(completed[1:3],key=selection_score)['parameters']
            params=dict(preferred,factors=128)
        else:
            preferred=max(completed[1:4],key=selection_score)['parameters']
            params=dict(preferred,regularization=0.01)
        name='A'+str(index); directory=root/'trials'/name; result_path=directory/'trial_result.json'
        expected=fingerprint(dict(source=signature,parameters=params))
        if result_path.exists():
            result=read_json(result_path)
            if result.get('status')!='complete' or result.get('signature')!=expected: raise ValueError('Trial checkpoint mismatch')
        else:
            users,items,fit=fit_als(fold,params,directory,signature,config)
            with Monitor(config['score_devices']) as monitor:
                predictions=als_predictions(fold,users,items,directory/'ranking',dict(source=signature,trial=name),config,only_warm=True)
            result=dict(status='complete',signature=expected,trial=name,parameters=params,**warm_score(fold,predictions),
                warm_queries=int(fold.warm.sum()),fit_resources=fit['resources'],ranking_resources=monitor.result)
            save_json(result_path,result)
            del users,items,predictions
        completed.append(result)
        best=max(completed,key=selection_score)
        save_json(root/'selection_progress.json',dict(best=best,trials=completed))
        # Only remove our completed nonwinning factor files; retain metrics/ranking for every trial.
        for old in completed:
            if old['trial']==best['trial']: continue
            model_dir=root/'trials'/old['trial']
            for name in ('user_factors.npy','item_factors.npy'):
                path=child(root,str(path_relative(model_dir/name,root)))
                if path.exists(): path.unlink()
            state_path=model_dir/'fit_manifest.json'
            if state_path.exists(): save_json(state_path,dict(read_json(state_path),discarded=True))
        print('Trial complete:',result['trial'],'NDCG@10=',result['ndcg@10'],'current best=',best['trial'],flush=True)
    best=max(completed,key=selection_score)
    save_json(root/'best_config.json',dict(status='locked',source=signature,best=best,selection='warm-full validation NDCG@10, Recall@10, fewer factors; seed fixed'))
    pd.DataFrame([dict(trial=r['trial'],**r['parameters'],**{k:r[k] for k in ('ndcg@10','recall@10','ndcg@20','recall@20')}) for r in completed]).to_csv(root/'trials.csv',index=False)
    return best


def path_relative(path, root):
    return Path(path).resolve().relative_to(Path(root).resolve())


def discover_gold(root=None):
    if root: return Path(root).resolve()
    paths=sorted(Path('/kaggle/input').rglob('dataset_manifest.json'))
    matches=[p.parent for p in paths if read_json(p).get('export_version')=='gold_training_export_v1']
    if len(matches)!=1: raise ValueError('Attach exactly one gold-training-v2 Dataset or set GOLD_ROOT explicitly')
    return matches[0]


def discover_embedding(gold, root=None, bundle_name=None):
    root=Path(root or '/kaggle/input')
    expected=sha256(gold.directory/'text_export_manifest.json')
    matches=[]
    for path in root.rglob('embedding_manifest.json'):
        data=read_json(path)
        if data.get('signature',{}).get('export_manifest_sha256')==expected and (bundle_name is None or path.parent.name==bundle_name):
            matches.append(path)
    if len(matches)!=1: raise ValueError('Attach the matching complete category embedding Dataset or set EMBEDDING_ROOT/BUNDLE_NAME. Matches: '+str([p.parent.name for p in matches]))
    if read_json(matches[0]).get('status')!='complete': raise ValueError('Finish the partial embedding Dataset before training')
    return matches[0]


def resume_output(output, previous, minimum):
    if not previous or not Path(previous).exists(): return
    previous=Path(previous).resolve()
    if previous==Path(output).resolve(): return
    files=[p for p in previous.rglob('*') if p.is_file() and p.suffix in ('.json','.npy','.npz','.parquet','.csv','.pt')]
    if any(p.is_symlink() for p in files): raise ValueError('Resume must not contain symlinks')
    files=[p for p in files if not child(output,p.relative_to(previous)).exists()]
    disk_guard(output.parent,sum(p.stat().st_size for p in files),minimum)
    for path in files:
        target=child(output,path.relative_to(previous))
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copyfile(path,target)


def compare_results(first, second, output, samples=1000, seed=42):
    first=Path(first); second=Path(second)
    a=read_json(first/'evaluation_manifest.json'); b=read_json(second/'evaluation_manifest.json')
    if a['status']!='complete' or b['status']!='complete' or (a['source'],a['fold'])!=(b['source'],b['fold']):
        raise ValueError('Compare only matching Gold/category/fold evaluation results')
    af=pd.read_parquet(first/(a['model']+'_popularity_per_query.parquet'))
    bf=pd.read_parquet(second/(b['model']+'_popularity_per_query.parquet'))
    merged=af.merge(bf,on='query_id',how='outer',suffixes=('_a','_b'),validate='one_to_one',indicator=True)
    if not (merged['_merge']=='both').all() or not (merged.warm_full_a==merged.warm_full_b).all(): raise ValueError('Comparison query populations differ')
    delta=merged['ndcg@10_b'].to_numpy()-merged['ndcg@10_a'].to_numpy()
    result=dict(direction=b['model']+' minus '+a['model'],all_queries=paired_ci(delta,samples,seed),
        warm_full=paired_ci(delta[merged.warm_full_a.to_numpy(bool)],samples,seed))
    save_json(output,result)
    return result


def run(config):
    hardware=preflight(config)
    gold=Gold(discover_gold(config.get('gold_root')),config['category'])
    emb_path=discover_embedding(gold,config.get('embedding_root'),config.get('embedding_bundle')) if config['algorithm']=='content' else None
    signature=dict(gold=gold.signature,algorithm=config['algorithm'],seed=config['seed'],versions=versions(),
        code_sha256=sha256(__file__),als_backend=config.get('als_backend') if config['algorithm']=='als' else None,
        cpu_threads=config['cpu_threads'],embedding_manifest=sha256(emb_path) if emb_path else None,
        scoring_backend='cuda' if config['score_devices'][0].startswith('cuda:') else 'numpy',
        score_shard_queries=config['score_shard_queries'],policy=PROTOCOL)
    parent=Path(config['output_root']).resolve(); parent.mkdir(parents=True,exist_ok=True)
    root=parent/(config['algorithm']+'_'+config['category']+'_'+fingerprint(signature)[:16])
    previous=Path(config['resume_root'])/root.name if config.get('resume_root') else None
    resume_output(root,previous,config['min_free_gib'])
    root.mkdir(parents=True,exist_ok=True)
    if (root/'run.json').exists() and read_json(root/'run.json')['signature']!=signature: raise ValueError('Run source/config mismatch')
    phase=config['phase']; phase_root=root/phase
    phase_root.mkdir(exist_ok=True)
    if phase=='test':
        if not (root/'validation_complete.json').is_file() or read_json(root/'validation_complete.json')['source']!=signature:
            raise ValueError('Run/restore validation first. Test needs locked validation results from this exact source/config.')
    complete=phase_root/'phase_complete.json'
    if complete.is_file():
        saved=read_json(complete)
        artifacts=saved.get('artifacts',{})
        if (saved.get('status')=='complete' and saved.get('source')==signature and saved.get('fold')==phase
                and artifacts and all(child(root,p).is_file() and sha256(child(root,p))==digest for p,digest in artifacts.items())):
            print('Reused complete phase:',phase,root,flush=True)
            return root
    save_json(root/'run.json',dict(status='running',phase=phase,signature=signature,config=config,hardware=hardware))
    started=time.monotonic()
    try:
        disk_guard(root,reserve_gib=config['min_free_gib'])
        with Monitor(config['score_devices']) as loading:
            fold=gold.fold(phase)
        save_json(phase_root/'load_resources.json',loading.result)
        fold.copy_mappings(phase_root)
        if config['algorithm']=='als':
            if phase=='validation': best=tune_als(fold,root,signature,config)
            else:
                selection=read_json(root/'best_config.json')
                if selection.get('status')!='locked' or selection['source']!=signature: raise ValueError('No matching locked ALS config')
                best=selection['best']
            model_root=root/'trials'/best['trial'] if phase=='validation' else phase_root/'model'
            users,items,state=fit_als(fold,best['parameters'],model_root,signature,config)
            with Monitor(config['score_devices']) as scoring:
                native=als_predictions(fold,users,items,phase_root/'ranking',dict(source=signature,model=state['hashes']),config)
            save_json(phase_root/'score_resources.json',scoring.result)
        else:
            with Monitor(config['score_devices']) as building:
                items,item_valid,emb_info=load_item_vectors(gold,fold,emb_path.parent,emb_path.parent.name)
                items,normalized=normalize_rows(items)
                if not np.array_equal(item_valid,normalized): raise ValueError('Content vector coverage changed')
                positions,profiles,profile_info=content_profiles(fold.binary,fold.user_indices,items,item_valid)
            save_json(phase_root/'content_profiles.json',dict(embedding=emb_info,profiles=profile_info,resources=building.result,
                policy='uniform mean of distinct train items; normalized; static metadata snapshot'))
            disk_guard(root,items.nbytes+profiles.nbytes,config['min_free_gib'])
            save_array(phase_root/'item_vectors.npy',items)
            save_array(phase_root/'query_profiles.npy',profiles)
            save_array(phase_root/'profile_query_positions.npy',positions)
            with Monitor(config['score_devices']) as scoring:
                native=np.full((fold.nqueries,20),-1,np.int32)
                native[positions]=rank_checkpoints(positions,profiles,items,item_valid,phase_root/'ranking',dict(source=signature),config)
            save_json(phase_root/'score_resources.json',scoring.result)
        with Monitor(config['score_devices']) as metrics_monitor:
            rows=evaluate(fold,native,phase_root/'evaluation',config['algorithm'],config)
        save_json(phase_root/'metric_resources.json',metrics_monitor.result)
        if phase=='validation': save_json(root/'validation_complete.json',dict(status='complete',source=signature))
        artifacts={p.relative_to(root).as_posix():sha256(p) for p in sorted(phase_root.rglob('*')) if p.is_file() and p!=complete}
        if phase=='validation':
            for name in ('validation_complete.json','best_config.json'):
                if (root/name).is_file(): artifacts[name]=sha256(root/name)
        save_json(complete,dict(status='complete',source=signature,fold=phase,
            seconds=round(time.monotonic()-started,3),artifacts=artifacts))
        save_json(root/'run.json',dict(status='complete',phase=phase,signature=signature,config=config,hardware=hardware))
        print(pd.DataFrame(rows).query("cohort in ['all_queries','warm_full']").to_string(index=False))
        print('Saved results:',root,flush=True)
        return root
    except BaseException as exc:
        save_json(root/'run.json',dict(status='failed',phase=phase,signature=signature,config=config,hardware=hardware,error_type=type(exc).__name__,error=str(exc)))
        raise


def configure_kaggle_auth():
    """Read optional Kaggle Secrets without writing credentials into configs or artifacts."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        secrets=None
    else:
        secrets=UserSecretsClient()
    if secrets is not None:
        for name in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):
            if os.environ.get(name): continue
            try: value=secrets.get_secret(name)
            except Exception: continue  # Optional; native auth is checked by the Dataset request below.
            if value: os.environ[name]=value
    # KaggleHub otherwise prioritizes the notebook session token file over env credentials.
    from kagglehub.config import set_kaggle_api_token, set_kaggle_credentials
    token=os.environ.get('KAGGLE_API_TOKEN','').strip()
    username=os.environ.get('KAGGLE_USERNAME','').strip()
    key=os.environ.get('KAGGLE_KEY','').strip()
    if token or key.startswith('KGAT_'):
        set_kaggle_api_token(token or key)
        print('Kaggle auth: explicit API token selected (value hidden).',flush=True)
    elif username and key:
        set_kaggle_credentials(username,key)
        print('Kaggle auth: explicit legacy credentials selected (values hidden).',flush=True)
    else:
        print('Kaggle auth: native/session configuration; no explicit API token or complete legacy pair found.',flush=True)


def owned_result_dataset(handle):
    """Resolve ambiguous metadata 403 using the authenticated account's own inventory."""
    from kagglehub.clients import build_kaggle_client
    from kagglesdk.datasets.types.dataset_api_service import ApiListDatasetsRequest, DatasetSelectionGroup
    owner=handle.split('/')[0].casefold()
    owner_verified=False; seen_pages=set(); token=''
    with build_kaggle_client() as api:
        for page in range(1,1001):
            request=ApiListDatasetsRequest()
            request.group=DatasetSelectionGroup.DATASET_SELECTION_GROUP_MY
            request.page=page; request.page_size=100; request.page_token=token
            result=api.datasets.dataset_api_client.list_datasets(request)
            datasets=result.datasets or []
            refs=tuple(sorted(d.ref.casefold() for d in datasets))
            if not refs: break
            if refs in seen_pages: raise RuntimeError('Kaggle owned Dataset pagination repeated; cannot establish absence')
            seen_pages.add(refs)
            for dataset in datasets:
                ref=dataset.ref.casefold()
                owner_verified |= ref.split('/')[0]==owner
                if ref==handle.casefold():
                    if dataset.is_private is not True: raise ValueError('Result Dataset must be private: '+handle)
                    return True
            next_token=result.next_page_token
            if token and not next_token: break
            token=next_token
        else:
            raise RuntimeError('Too many owned Dataset pages; cannot establish absence')
    if not owner_verified:
        raise RuntimeError('Cannot verify RESULT_OWNER against authenticated owned Datasets: '+handle)
    print('Result Dataset absent from verified owned inventory; API will create private Dataset:',handle,flush=True)
    return False


def result_dataset_private(handle):
    """Metadata 403 is ambiguous; verify owned inventory before deciding to create."""
    if not isinstance(handle,str) or not re.fullmatch(r'[A-Za-z0-9_-]+/[A-Za-z0-9_-]+',handle):
        raise ValueError('Result Dataset must be owner/slug without a version')
    from kagglehub.clients import BackendError, build_kaggle_client
    from requests.exceptions import HTTPError
    from kagglesdk.datasets.types.dataset_api_service import ApiGetDatasetMetadataRequest
    request=ApiGetDatasetMetadataRequest()
    request.owner_slug,request.dataset_slug=handle.split('/')
    try:
        with build_kaggle_client() as api:
            response=api.datasets.dataset_api_client.get_dataset_metadata(request)
    except (BackendError,HTTPError) as exc:
        response=getattr(exc,'response',None)
        status=response.status_code if response is not None else getattr(exc,'error_code',None)
        if status==404: return False
        if status==403:
            try: return owned_result_dataset(handle)
            except ValueError: raise
            except Exception:
                raise RuntimeError('Kaggle metadata HTTP 403 and owned inventory could not verify access/absence for '+handle+
                    '. Check token and RESULT_OWNER; no upload attempted.') from None
        raise RuntimeError(f'Cannot verify result Dataset {handle} (HTTP {status}). Check Internet, Kaggle Secrets/API permissions; no upload attempted.') from None
    if response.info.is_private is not True:
        raise ValueError('Result Dataset must be private: '+handle)
    return True


def publish_results(root, handle, minimum=5, previous=None):
    """Snapshot one complete category run; API failures never delete local results."""
    import kagglehub
    root=Path(root).resolve()
    state=read_json(root/'run.json')
    for phase in BOUNDS:
        done=read_json(root/phase/'phase_complete.json')
        if done.get('status')!='complete' or done.get('source')!=state['signature']:
            raise ValueError('Upload requires matching complete validation and test')
        if not done.get('artifacts') or any(not child(root,p).is_file() or sha256(child(root,p))!=digest
                for p,digest in done['artifacts'].items()):
            raise ValueError('Completed phase artifacts changed; rerun the phase before upload')
    allowed={'.json','.npy','.npz','.parquet','.csv','.pt'}
    files=[p for p in sorted(root.rglob('*')) if p.is_file() and p.suffix in allowed]
    if any(p.is_symlink() for p in root.rglob('*')): raise ValueError('Result snapshot must not contain symlinks')
    inventory={p.relative_to(root).as_posix():dict(bytes=p.stat().st_size,sha256=sha256(p)) for p in files}
    bundle=dict(format='gold_recommendation_results_v1',handle=handle,run=root.name,
        source=state['signature'],phases=list(BOUNDS),files=inventory)
    digest=fingerprint(bundle)
    receipt_path=root.parent/(root.name+'_upload_receipt.json')
    if receipt_path.is_file():
        receipt=read_json(receipt_path)
        if receipt.get('handle')==handle and receipt.get('bundle_sha256')==digest:
            if receipt.get('status')=='submitted_processing':
                print('Upload already submitted:',handle,flush=True)
                return receipt
            raise RuntimeError('Previous upload status is uncertain. Check the Dataset before removing receipt to retry: '+str(receipt_path))
    if previous and previous.get('bundle_sha256')==digest:
        receipt=dict(status='submitted_processing',handle=handle,bundle_sha256=digest,reused_remote=True)
        save_json(receipt_path,receipt)
        return receipt
    result_dataset_private(handle)
    # Reserve snapshot plus SDK archive space; keep only this category in the upload.
    disk_guard(root.parent,2*sum(v['bytes'] for v in inventory.values()),minimum)
    with tempfile.TemporaryDirectory(prefix='recommendation-upload-',dir=root.parent) as temporary:
        staging=Path(temporary)
        for path in files:
            target=staging/'recommendation_results'/root.name/path.relative_to(root)
            target.parent.mkdir(parents=True,exist_ok=True)
            shutil.copyfile(path,target)
        save_json(staging/'result_bundle.json',dict(bundle,bundle_sha256=digest))
        receipt=dict(status='upload_started',handle=handle,bundle_sha256=digest)
        save_json(receipt_path,receipt)
        try:
            kagglehub.dataset_upload(handle,str(staging),version_notes='Validation + locked test; '+digest[:16])
        except BaseException:
            save_json(receipt_path,dict(receipt,status='upload_uncertain'))
            raise RuntimeError('Kaggle upload interrupted; local results preserved. Check remote version before retrying: '+handle) from None
    receipt=dict(receipt,status='submitted_processing',url='https://www.kaggle.com/datasets/'+handle)
    save_json(receipt_path,receipt)
    print('Results uploaded; Kaggle processing may still be pending:',receipt['url'],flush=True)
    return receipt


def run_suite(config):
    """Validation -> locked test -> private Dataset, sequentially for each category."""
    categories=config['categories']
    if not categories or len(set(categories))!=len(categories) or any(c not in CATEGORIES for c in categories):
        raise ValueError('Select unique supported categories')
    publish=config.get('publish_results',True)
    handles=config.get('result_datasets',{})
    if publish:
        if len({handles.get(c) for c in categories})!=len(categories): raise ValueError('Use a separate result Dataset per category')
        configure_kaggle_auth()
        remote={c:result_dataset_private(handles[c]) for c in categories}
    # Validate every selected embedding before starting a long first category.
    for category in categories:
        gold=Gold(discover_gold(config.get('gold_root')),category)
        if config['algorithm']=='content': discover_embedding(gold,config.get('embedding_root'),config.get('embedding_bundle'))
    parent=Path(config['output_root']);parent.mkdir(parents=True,exist_ok=True)
    summary=[]
    for category in categories:
        current=dict(config,category=category)
        previous=None
        if publish and remote[category] and config.get('auto_resume',True):
            import kagglehub
            restored=Path(kagglehub.dataset_download(handles[category]))
            bundle=read_json(restored/'result_bundle.json')
            if bundle.get('format')!='gold_recommendation_results_v1' or bundle.get('handle')!=handles[category]:
                raise ValueError('Unexpected result Dataset layout: '+handles[category])
            previous=bundle
            current['resume_root']=str(restored/'recommendation_results')
        for phase in BOUNDS:
            print('\n===',config['algorithm'],category,phase,'===',flush=True)
            root=run(dict(current,phase=phase))
        receipt=publish_results(root,handles[category],config['min_free_gib'],previous) if publish else None
        summary.append(dict(category=category,run_root=str(root),dataset=handles.get(category),upload=receipt))
        save_json(parent/(config['algorithm']+'_suite_results.json'),summary)
    return summary


if __name__=='__main__':
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--config',required=True)
    parser.add_argument('--suite',action='store_true')
    arguments=parser.parse_args()
    (run_suite if arguments.suite else run)(read_json(arguments.config))


In [ ]:
%%writefile /kaggle/working/gold_neural_recommendation.py
"""Standalone Gold NeuMF / DeepFM training, temporal features and resumable exact ranking."""
import argparse
import copy
import gc
import hashlib
import json
import os
from pathlib import Path
import shutil
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from contextlib import closing

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from torch import nn

try:
    import gold_recommendation as rec  # Generated standalone Kaggle notebook.
except ModuleNotFoundError as error:
    if error.name != 'gold_recommendation':
        raise
    import kaggle_recommendation as rec

FEATURE_SCHEMA = ['rating_count_valid', 'mean_rating', 'low_rating_fraction',
                  'high_rating_fraction', 'review_text_count', 'rating_missing',
                  'review_missing']
VARIANTS = ('NCF_ID', 'D0', 'D1', 'D2')
TOWER_VARIANTS = ('TT0', 'TT1', 'TT2')
DEFAULT_TRIALS = [dict(name='C8', dim=8, hidden=[32, 16], lr=.001, weight_decay=.00001)]
TRAINING_SIGNATURE_KEYS = ('training_profile', 'seed', 'trials', 'max_epochs', 'eval_epochs',
    'early_stopping', 'early_stop_start_epoch', 'early_stop_patience', 'early_stop_min_delta',
    'negatives', 'projection_dim', 'train_batch', 'train_device', 'score_devices', 'cpu_threads',
    'bootstrap_samples', 'score_shard_queries', 'adam_foreach', 'prefetch_train', 'score_feature_cache')
SAMPLER_VERSION = 'stateless_splitmix64_uniform_v1'


def default_config(algorithm):
    if algorithm == 'two_tower':
        config = default_config('deepfm')
        config.update(algorithm=algorithm, variants=list(TOWER_VARIANTS),
            output_root='/kaggle/working/two_tower_results', query_batch=512,
            item_block=16384, score_shard_queries=2048, train_batch=20480,
            tower_output_dim=32, publish_results=False,
            trials=[dict(name='T8', dim=8, hidden=[64], lr=.001, weight_decay=.00001)])
        return config
    if algorithm not in ('ncf', 'deepfm'):
        raise ValueError('algorithm must be ncf, deepfm or two_tower')
    return dict(algorithm=algorithm, training_profile='compact_full_users',
        variants=['NCF_ID'] if algorithm == 'ncf' else ['D0', 'D1', 'D2'],
        categories=list(rec.CATEGORIES), gold_root=None, embedding_root=None, embedding_bundle=None,
        output_root='/kaggle/working/neural_results', resume_root=None, seed=42, cpu_threads=2,
        train_device='cuda:0', score_devices=['cuda:0', 'cuda:1'], train_batch=4096,
        query_batch=32, item_block=8192, pair_batch=65536, score_shard_queries=128,
        adam_foreach=True, prefetch_train=True, score_feature_cache=True,
        max_epochs=20, eval_epochs=[1, 3, 6, 9, 12, 15, 18, 20],
        early_stopping=True, early_stop_start_epoch=6, early_stop_patience=2,
        early_stop_min_delta=.00001, negatives=4, trials=copy.deepcopy(DEFAULT_TRIALS),
        projection_dim=16, bootstrap_samples=1000, min_free_gib=5,
        publish_results=True, auto_resume=True, phases=['validation', 'test'],
        result_datasets={}, dataset_versions={'gold': None, 'embedding': None})


def validate_config(config):
    if config['algorithm'] not in ('ncf', 'deepfm', 'two_tower'):
        raise ValueError('Unknown neural algorithm')
    if config['training_profile'] != 'compact_full_users':
        raise ValueError('Only compact_full_users is implemented; keep the full Gold population')
    if type(config['early_stopping']) is not bool:
        raise ValueError('early_stopping must be boolean')
    for key in ('adam_foreach', 'prefetch_train', 'score_feature_cache'):
        if type(config[key]) is not bool:
            raise ValueError(key + ' must be boolean')
    for key in ('early_stop_start_epoch', 'early_stop_patience'):
        if type(config[key]) is not int or config[key] < 1:
            raise ValueError('Positive integer required: ' + key)
    if (isinstance(config['early_stop_min_delta'], bool)
            or not isinstance(config['early_stop_min_delta'], (int, float))
            or not np.isfinite(config['early_stop_min_delta']) or config['early_stop_min_delta'] < 0):
        raise ValueError('early_stop_min_delta must be finite and nonnegative')
    variants = config['variants']
    allowed = {'ncf': ('NCF_ID',), 'deepfm': ('D0', 'D1', 'D2'), 'two_tower': TOWER_VARIANTS}[config['algorithm']]
    if config['algorithm'] == 'two_tower' and (type(config.get('tower_output_dim')) is not int or config['tower_output_dim'] < 1):
        raise ValueError('Positive tower_output_dim required')
    if not variants or len(set(variants)) != len(variants) or any(v not in allowed for v in variants):
        raise ValueError('Invalid or duplicate variants')
    categories = config['categories']
    if not categories or len(set(categories)) != len(categories) or any(c not in rec.CATEGORIES for c in categories):
        raise ValueError('Invalid or duplicate categories')
    if config['phases'] not in (['validation'], ['validation', 'test'], ['test']):
        raise ValueError('phases must be [validation], [validation,test], or [test] with restored validation')
    for key in ('train_batch', 'query_batch', 'item_block', 'pair_batch', 'score_shard_queries',
                'max_epochs', 'cpu_threads', 'bootstrap_samples', 'projection_dim', 'negatives'):
        if type(config[key]) is not int or config[key] < 1:
            raise ValueError('Positive integer required: ' + key)
    if type(config['seed']) is not int or not 0 <= config['seed'] < 2**32:
        raise ValueError('seed must be uint32')
    epochs = config['eval_epochs']
    if not epochs or sorted(set(epochs)) != epochs or any(type(e) is not int or e < 1 or e > config['max_epochs'] for e in epochs) or epochs[-1] != config['max_epochs']:
        raise ValueError('Invalid evaluation epoch schedule')
    if config['early_stopping'] and config['early_stop_start_epoch'] > config['max_epochs']:
        raise ValueError('Early-stop start epoch exceeds max_epochs; adjust the budget or disable early stopping explicitly')
    trials = config['trials']
    if not trials or len({p['name'] for p in trials}) != len(trials):
        raise ValueError('Need unique trials')
    for trial in trials:
        if not trial['name'].isalnum() or type(trial['dim']) is not int or trial['dim'] < 1:
            raise ValueError('Invalid trial name/dimension')
        if not trial['hidden'] or any(type(h) is not int or h < 1 for h in trial['hidden']):
            raise ValueError('Invalid MLP dimensions')
        if not np.isfinite(trial['lr']) or trial['lr'] <= 0 or not np.isfinite(trial['weight_decay']) or trial['weight_decay'] < 0:
            raise ValueError('Invalid optimizer parameters')
    devices = config['score_devices']
    if not devices or len(set(devices)) != len(devices) or ('cpu' in devices and devices != ['cpu']):
        raise ValueError('Use unique CUDA score devices or [cpu]')
    torch.set_num_threads(config['cpu_threads'])
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True)
    for device in set(devices + [config['train_device']]):
        if device == 'cpu':
            continue
        if not device.startswith('cuda:') or not device[5:].isdigit():
            raise ValueError('Explicit device required: cpu or cuda:N')
        if not torch.cuda.is_available() or int(device[5:]) >= torch.cuda.device_count():
            raise RuntimeError('Requested ' + device + ' unavailable. Enable Kaggle GPU or explicitly select cpu; no silent fallback.')
    if any(d.startswith('cuda:') for d in devices + [config['train_device']]):
        os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')


def seed_model(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class PairNetwork(nn.Module):
    """NeuMF uses separate GMF/MLP tables; DeepFM shares ID tables across FM/Deep."""
    def __init__(self, nusers, nitems, variant, parameters, projection_dim=16):
        super().__init__()
        if variant not in VARIANTS:
            raise ValueError('Unknown variant')
        self.variant = variant
        dim = parameters['dim']
        self.user = nn.Embedding(nusers, dim)
        self.item = nn.Embedding(nitems, dim)
        width = 2 * dim
        if variant == 'NCF_ID':
            self.gmf_user = nn.Embedding(nusers, dim)
            self.gmf_item = nn.Embedding(nitems, dim)
        else:
            self.user_bias = nn.Embedding(nusers, 1)
            self.item_bias = nn.Embedding(nitems, 1)
            self.bias = nn.Parameter(torch.zeros(1))
        if variant in ('D1', 'D2'):
            self.item_projection = nn.Linear(384, projection_dim)
            width += projection_dim + 1
        if variant == 'D2':
            self.review_projection = nn.Linear(384, projection_dim)
            width += projection_dim + len(FEATURE_SCHEMA)
        layers = []
        for hidden in parameters['hidden']:
            layers.extend([nn.Linear(width, hidden), nn.ReLU()])
            width = hidden
        self.deep = nn.Sequential(*layers)
        self.output = nn.Linear(width + (dim if variant == 'NCF_ID' else 0), 1)
        for module in self.modules():
            if isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=.01)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)
        if variant != 'NCF_ID':
            nn.init.zeros_(self.user_bias.weight)
            nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items, item_text=None, item_present=None, stats=None, reviews=None,
                item_projected=None, review_projected=None):
        u, i = self.user(users), self.item(items)
        fields = [u, i]
        if self.variant in ('D1', 'D2'):
            fields.extend([torch.relu(self.item_projection(item_text)) if item_projected is None else item_projected,
                           item_present[:, None]])
        if self.variant == 'D2':
            fields.extend([stats, torch.relu(self.review_projection(reviews)) if review_projected is None else review_projected])
        deep = self.deep(torch.cat(fields, dim=1))
        if self.variant == 'NCF_ID':
            product = self.gmf_user(users) * self.gmf_item(items)
            return self.output(torch.cat([product, deep], dim=1)).squeeze(1)
        return (self.output(deep).squeeze(1) + (u * i).sum(dim=1)
                + self.user_bias(users).squeeze(1) + self.item_bias(items).squeeze(1) + self.bias)


class TwoTowerNetwork(nn.Module):
    """Independent encoders; frozen MiniLM inputs, raw dot-product output."""
    def __init__(self, nusers, nitems, variant, parameters, projection_dim=16, output_dim=32):
        super().__init__()
        if variant not in TOWER_VARIANTS:
            raise ValueError('Unknown Two-Tower variant')
        self.variant, self.output_dim = variant, output_dim
        dim = parameters['dim']
        self.user = nn.Embedding(nusers, dim)
        self.item = nn.Embedding(nitems, dim)
        user_width = item_width = dim
        if variant in ('TT1', 'TT2'):
            self.item_projection = nn.Linear(384, projection_dim)
            item_width += projection_dim + 1
        if variant == 'TT2':
            self.review_projection = nn.Linear(384, projection_dim)
            user_width += projection_dim + len(FEATURE_SCHEMA)
        def tower(width):
            layers = []
            for hidden in parameters['hidden']:
                layers.extend([nn.Linear(width, hidden), nn.ReLU()])
                width = hidden
            return nn.Sequential(*layers, nn.Linear(width, output_dim))
        self.user_tower, self.item_tower = tower(user_width), tower(item_width)
        for module in self.modules():
            if isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=.01)
            elif isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def encode_user(self, users, stats=None, reviews=None):
        fields = [self.user(users)]
        if self.variant == 'TT2':
            fields.extend([self.review_projection(reviews), stats])
        return self.user_tower(torch.cat(fields, dim=1))

    def encode_item(self, items, item_text=None, item_present=None):
        fields = [self.item(items)]
        if self.variant in ('TT1', 'TT2'):
            fields.extend([self.item_projection(item_text), item_present[:, None]])
        return self.item_tower(torch.cat(fields, dim=1))

    def forward(self, users, items, item_text=None, item_present=None, stats=None, reviews=None):
        return (self.encode_user(users, stats, reviews) * self.encode_item(items, item_text, item_present)).sum(dim=1)


def make_model(nusers, nitems, variant, parameters, config):
    if variant in TOWER_VARIANTS:
        return TwoTowerNetwork(nusers, nitems, variant, parameters, config['projection_dim'], config['tower_output_dim'])
    return PairNetwork(nusers, nitems, variant, parameters, config['projection_dim'])


def load_examples(gold, fold):
    frame = gold.table(fold.name + '/train_examples',
        ['example_id', 'user_idx', 'item_idx', 'first_pair_time', 'prediction_time', 'strength'])
    if not len(frame) or frame.isna().any().any() or frame.example_id.duplicated().any() or len(frame) != fold.binary.nnz:
        raise ValueError('Missing/duplicate positive anchors')
    if frame.duplicated(['user_idx', 'item_idx']).any() or not (frame.strength == 1).all():
        raise ValueError('Need exactly one positive per train pair')
    if not (frame.prediction_time == frame.first_pair_time).all() or (frame.prediction_time >= pd.Timestamp(fold.start)).any():
        raise ValueError('Invalid positive anchor time')
    users = frame.user_idx.to_numpy(np.int64)
    items = frame.item_idx.to_numpy(np.int64)
    if (users < 0).any() or (users >= fold.nusers).any() or (items < 0).any() or (items >= fold.nitems).any():
        raise ValueError('Positive indices out of mapping')
    if not np.all(np.asarray(fold.binary[users, items]).ravel() == 1):
        raise ValueError('Positive anchors differ from train pairs')
    pairs = gold.table(fold.name + '/train_pairs', ['user_idx', 'item_idx', 'first_pair_time'])
    checked = frame.merge(pairs, on=['user_idx', 'item_idx'], validate='one_to_one', suffixes=('', '_pair'))
    if len(checked) != len(frame) or not (checked.first_pair_time == checked.first_pair_time_pair).all():
        raise ValueError('Positive first_pair_time mismatch')
    frame = frame.sort_values('example_id', kind='stable').reset_index(drop=True)
    frame['sampler_key'] = np.fromiter((int.from_bytes(hashlib.sha256(str(x).encode()).digest()[:8], 'little')
                                       for x in frame.example_id), dtype=np.uint64, count=len(frame))
    return frame


def sample_negatives(binary, user, key, epoch, seed, count):
    values = sample_negative_batch(binary, np.array([user]), np.array([key], np.uint64), epoch, seed, count)[0]
    return values[values >= 0]


def mix64(values):
    """SplitMix64 finalizer, vectorized over independent example counters."""
    with np.errstate(over='ignore'):
        values = np.asarray(values, np.uint64) + np.uint64(0x9E3779B97F4A7C15)
        values = (values ^ (values >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
        values = (values ^ (values >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
        return values ^ (values >> np.uint64(31))


def observed_pair_keys(binary):
    if binary.shape[0] * binary.shape[1] > np.iinfo(np.int64).max:
        raise ValueError('User/item product exceeds int64 sampling keys')
    rows = np.repeat(np.arange(binary.shape[0], dtype=np.int64), np.diff(binary.indptr))
    keys = rows * binary.shape[1] + binary.indices
    return keys if binary.has_sorted_indices else np.sort(keys)


def sample_negative_batch(binary, users, keys, epoch, seed, count, observed=None):
    """Uniform train-only negatives; deterministic per example, independent of batching/prefetch."""
    users = np.asarray(users, np.int64)
    keys = np.asarray(keys, np.uint64)
    if users.shape != keys.shape or users.ndim != 1 or (users < 0).any() or (users >= binary.shape[0]).any():
        raise ValueError('Invalid negative-sampling anchors')
    nitems = binary.shape[1]
    if nitems < 1 or count < 1:
        raise ValueError('Positive candidate and negative counts required')
    observed = observed_pair_keys(binary) if observed is None else observed
    support = binary.indptr[users + 1] - binary.indptr[users]
    take = np.minimum(count, nitems - support)
    output = np.full((len(users), count), -1, np.int64)
    dense = (nitems - support <= 2 * count) | (support > nitems // 2)
    # Exact complement fallback prevents rejection loops for small/saturated pools.
    for row in np.flatnonzero(dense & (take > 0)):
        user = users[row]
        seen = binary.indices[binary.indptr[user]:binary.indptr[user + 1]]
        pool = np.setdiff1d(np.arange(nitems), seen, assume_unique=True)
        rng = np.random.default_rng([seed, epoch, int(keys[row]) & 0xffffffff, int(keys[row]) >> 32])
        output[row, :take[row]] = rng.choice(pool, size=take[row], replace=False)
    base = keys ^ mix64(np.uint64(seed) ^ np.uint64(0xA0761D6478BD642F)) ^ mix64(np.uint64(epoch) ^ np.uint64(0xE7037ED1A0B428DB))
    counters = np.zeros(len(users), np.uint64)
    # Accept an exact multiple of nitems outcomes; reject the uint64 tail to avoid modulo bias.
    limit = np.uint64(((2**64 - 1) // nitems) * nitems)
    for slot in range(count):
        pending = np.flatnonzero(~dense & (take > slot))
        while len(pending):
            with np.errstate(over='ignore'):
                random = mix64(base[pending] + counters[pending] * np.uint64(0x9E3779B97F4A7C15))
            counters[pending] += np.uint64(1)
            candidate = (random % np.uint64(nitems)).astype(np.int64)
            positions = np.searchsorted(observed, users[pending] * nitems + candidate)
            blocked = np.zeros(len(pending), bool)
            valid = positions < len(observed)
            blocked[valid] = observed[positions[valid]] == users[pending[valid]] * nitems + candidate[valid]
            if slot:
                blocked |= (output[pending, :slot] == candidate[:, None]).any(axis=1)
            accepted = ~blocked & (random < limit)
            output[pending[accepted], slot] = candidate[accepted]
            pending = pending[~accepted]
    return output


def epoch_batches(examples, binary, epoch, config):
    rng = np.random.default_rng([config['seed'], epoch])
    order = rng.permutation(len(examples))
    us = examples.user_idx.to_numpy(np.int64)
    it = examples.item_idx.to_numpy(np.int64)
    keys = examples.sampler_key.to_numpy(np.uint64)
    observed = observed_pair_keys(binary)
    # Keep feature references, never repeat dense D2 profiles for all sampled negatives on disk.
    anchors_per_batch = max(1, config['train_batch'] // (config['negatives'] + 1))
    for start in range(0, len(order), anchors_per_batch):
        anchors = order[start:start + anchors_per_batch]
        negative = sample_negative_batch(binary, us[anchors], keys[anchors], epoch,
                                         config['seed'], config['negatives'], observed)
        items = np.column_stack([it[anchors], negative])
        valid = items >= 0
        labels = np.zeros(items.shape, np.float32); labels[:, 0] = 1
        yield (np.broadcast_to(us[anchors, None], items.shape)[valid], items[valid], labels[valid],
               np.broadcast_to(anchors[:, None], items.shape)[valid])


def load_review_vectors(gold, manifest_path, wanted, directory, config):
    """Load only required text IDs into disk-backed rows, validating every shard like content."""
    manifest = rec.read_json(manifest_path)
    signature = manifest['signature']
    encoder = gold.contract['encoder']
    if manifest.get('status') != 'complete' or signature.get('export_manifest_sha256') != rec.sha256(gold.directory / 'text_export_manifest.json'):
        raise ValueError('Wrong/incomplete review embedding source')
    for key in ('model', 'revision', 'dimension', 'max_seq_length', 'normalize_embeddings', 'output_dtype', 'long_text_policy'):
        if signature.get(key) != encoder[key]:
            raise ValueError('Review embedding encoder mismatch: ' + key)
    if encoder['dimension'] != 384 or signature.get('cleaning_version') != 'gold_text_nfc_html_whitespace_v1':
        raise ValueError('Wrong review encoder dimension/cleaning')
    ids = sorted(wanted)
    lookup = {key: row for row, key in enumerate(ids)}
    directory.mkdir(parents=True, exist_ok=True)
    rec.disk_guard(directory, max(1, len(ids)) * 384 * 4, config['min_free_gib'])
    vectors = np.lib.format.open_memmap(directory / 'review_vectors.npy', mode='w+', dtype=np.float32,
                                        shape=(max(1, len(ids)), 384))
    vectors[:] = 0
    found = np.zeros(len(ids), bool)
    shards = manifest['shards']
    if len({s['path'] for s in shards}) != len(shards) or len(shards) != manifest['total_shards'] or sum(s['rows'] for s in shards) != manifest['texts']:
        raise ValueError('Incomplete/duplicate review shard list')
    for number, state in enumerate(shards, 1):
        path = rec.child(manifest_path.parent, state['path'])
        if not state['path'].startswith('texts/') or path.suffix != '.parquet' or state.get('signature') != signature or state.get('status') != 'complete':
            raise ValueError('Unexpected review embedding shard')
        if rec.read_json(path.with_suffix('.json')) != {k: v for k, v in state.items() if k != 'path'} or rec.sha256(path) != state['output_sha256']:
            raise ValueError('Review embedding checksum/checkpoint mismatch')
        parquet = pq.ParquetFile(path)
        if parquet.metadata.num_rows != state['rows']:
            raise ValueError('Review embedding row count mismatch')
        for batch in parquet.iter_batches(batch_size=1024, columns=['text_id', 'embedding']):
            batch_ids = batch.column(0).to_pylist()
            selected = [i for i, key in enumerate(batch_ids) if key in lookup]
            if not selected:
                continue
            picked = batch.take(pa.array(selected, pa.int32()))
            column = picked.column(1)
            if not pa.types.is_fixed_size_list(column.type) or column.type.list_size != 384 or column.type.value_type != pa.float32() or column.null_count or column.flatten().null_count:
                raise ValueError('Invalid review vector schema')
            values = column.flatten().to_numpy(zero_copy_only=False).reshape(-1, 384)
            if not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values, axis=1), 1, atol=.002):
                raise ValueError('Invalid review embedding norm')
            for key, vector in zip(picked.column(0).to_pylist(), values):
                row = lookup[key]
                if found[row]:
                    raise ValueError('Duplicate required review text_id')
                vectors[row] = vector
                found[row] = True
        if number == 1 or number % 20 == 0 or number == len(shards):
            print('Review embedding shards:', number, '/', len(shards), 'matched', int(found.sum()), '/', len(ids), flush=True)
    if not found.all():
        raise ValueError('Required review vector missing from complete bundle')
    vectors.flush()
    return vectors, lookup


def prefix_features(events, anchors, vectors, stats_out, reviews_out):
    """One user at a time; emit before consuming equal-time events. Repeated events retain weight."""
    events = events.sort_values(['user_idx', 'event_time', 'record_hash'], kind='stable').reset_index(drop=True)
    anchors = anchors.sort_values(['user_idx', 'prediction_time'], kind='stable')
    event_users = events.user_idx.to_numpy(np.int64)
    event_times = events.event_time.to_numpy(dtype='datetime64[ns]')
    rating = events.rating.to_numpy(np.float64, na_value=np.nan)
    vector_rows = events.vector_row.to_numpy(np.int64)
    anchor_users = anchors.user_idx.to_numpy(np.int64)
    times = anchors.prediction_time.to_numpy(dtype='datetime64[ns]')
    destinations = anchors.feature_row.to_numpy(np.int64)
    pos = 0
    while pos < len(anchors):
        user = anchor_users[pos]
        end = np.searchsorted(anchor_users, user, side='right')
        lo = np.searchsorted(event_users, user, side='left')
        hi = np.searchsorted(event_users, user, side='right')
        event = lo
        valid_count = text_count = low_count = high_count = 0
        rating_sum = 0.
        vector_sum = np.zeros(384, np.float64)
        for anchor in range(pos, end):
            while event < hi and event_times[event] < times[anchor]:
                value = rating[event]
                if np.isfinite(value):
                    if not 1 <= value <= 5:
                        raise ValueError('Historical rating outside 1..5')
                    valid_count += 1
                    rating_sum += value
                    low_count += int(value <= 2)
                    high_count += int(value >= 4)
                if vector_rows[event] >= 0:
                    vector_sum += vectors[vector_rows[event]]
                    text_count += 1
                event += 1
            row = destinations[anchor]
            norm = np.linalg.norm(vector_sum)
            usable = text_count > 0 and norm > 1e-12
            reviews_out[row] = vector_sum / norm if usable else 0
            stats_out[row] = [valid_count, rating_sum / valid_count if valid_count else 3.,
                low_count / valid_count if valid_count else 0., high_count / valid_count if valid_count else 0.,
                text_count, float(valid_count == 0), float(not usable)]
        pos = end


def feature_arrays(root):
    return {name: np.load(root / (name + '.npy'), mmap_mode='r')
            for name in ('stats', 'reviews', 'train_feature_rows', 'query_feature_rows', 'scaler_mean', 'scaler_scale')}


def build_d2(gold, fold, examples, manifest_path, root, config):
    root.mkdir(parents=True, exist_ok=True)
    signature = rec.fingerprint(dict(source=gold.signature, fold=fold.name, embedding=rec.sha256(manifest_path),
        examples=rec.array_hash(examples.sampler_key.to_numpy()), schema=FEATURE_SCHEMA, code=rec.sha256(__file__)))
    complete = root / 'features.json'
    if complete.is_file():
        state = rec.read_json(complete)
        if state.get('signature') != signature:
            raise ValueError('D2 feature cache source mismatch')
        if state.get('status') == 'complete' and not state.get('cache_compacted'):
            if any(not (root / name).is_file() or rec.sha256(root / name) != digest for name, digest in state['files'].items()):
                raise ValueError('D2 feature cache checksum mismatch')
            return feature_arrays(root)
    rec.save_json(complete, dict(status='building', signature=signature))
    for name in ('query_stats.npy', 'query_reviews.npy'):
        (root / name).unlink(missing_ok=True)
    events = gold.table(fold.name + '/history_events', ['record_hash', 'user_id', 'event_time'])
    if events.isna().any().any() or events.record_hash.duplicated().any() or (events.event_time >= pd.Timestamp(fold.start)).any():
        raise ValueError('Invalid/future historical events')
    users = gold.table(fold.name + '/users', ['user_id', 'user_idx'])
    events = events.merge(users, on='user_id', how='left', validate='many_to_one')
    if events.user_idx.isna().any():
        raise ValueError('Historical user absent from train mapping')
    refs = gold.table('review_text_refs', ['record_hash', 'user_id', 'event_time', 'text_id', 'has_text'])
    ratings = gold.table('history_ratings', ['record_hash', 'rating'])
    if refs.record_hash.duplicated().any() or ratings.record_hash.duplicated().any() or not np.array_equal(refs.text_id.notna(), refs.has_text):
        raise ValueError('Invalid review refs/rating lookup')
    events = events.merge(refs, on='record_hash', how='left', validate='one_to_one', suffixes=('', '_ref'))
    if events.has_text.isna().any() or not (events.user_id == events.user_id_ref).all() or not (events.event_time == events.event_time_ref).all():
        raise ValueError('Historical review refs mismatch')
    events = events.merge(ratings, on='record_hash', how='left', validate='one_to_one', indicator=True)
    if not (events['_merge'] == 'both').all():
        raise ValueError('Missing historical rating lookup row')
    anchors = pd.concat([examples[['user_idx', 'prediction_time']],
        pd.DataFrame(dict(user_idx=fold.user_indices, prediction_time=pd.Timestamp(fold.start)))], ignore_index=True)
    keys = pd.MultiIndex.from_frame(anchors)
    codes, unique = pd.factorize(keys, sort=True)
    unique_anchors = unique.to_frame(index=False)
    unique_anchors.columns = ['user_idx', 'prediction_time']
    # Row 0 is the shared canonical empty-prefix row; only anchors after a user's first event need distinct rows.
    first_event_time = events.groupby('user_idx', sort=False)['event_time'].min()
    has_prior = ((unique_anchors.user_idx >= 0)
                 & (unique_anchors.prediction_time > unique_anchors.user_idx.map(first_event_time))).to_numpy(bool)
    active_anchors = unique_anchors.loc[has_prior].copy()
    active_anchors['feature_row'] = np.arange(1, len(active_anchors) + 1, dtype=np.int64)
    anchor_rows = np.zeros(len(unique_anchors), dtype=np.int64)
    anchor_rows[has_prior] = active_anchors['feature_row'].to_numpy()
    feature_codes = anchor_rows[codes]
    n_feature_rows = len(active_anchors) + 1
    max_anchor_time = active_anchors.drop_duplicates('user_idx', keep='last').set_index('user_idx')['prediction_time']
    consumed_text = events.has_text & (events.event_time < events.user_idx.map(max_anchor_time))
    wanted = set(events.loc[consumed_text, 'text_id'])
    anchor_bytes = n_feature_rows * (384 + len(FEATURE_SCHEMA)) * 4
    vector_bytes = max(1, len(wanted)) * 384 * 4
    temp_base = Path(tempfile.gettempdir())
    use_temp_vectors = temp_base.exists() and shutil.disk_usage(temp_base).free > shutil.disk_usage(root).free
    rec.disk_guard(root, anchor_bytes + (0 if use_temp_vectors else vector_bytes), config['min_free_gib'])
    temp_context = tempfile.TemporaryDirectory(prefix='d2-vectors-', dir=temp_base) if use_temp_vectors else None
    try:
        vector_root = Path(temp_context.name) if temp_context is not None else root
        vectors, lookup = load_review_vectors(gold, manifest_path, wanted, vector_root, config)
        events['vector_row'] = events.text_id.map(lookup).fillna(-1).astype(np.int64)
        stats = np.lib.format.open_memmap(root / 'stats.npy', mode='w+', dtype=np.float32, shape=(n_feature_rows, len(FEATURE_SCHEMA)))
        reviews = np.lib.format.open_memmap(root / 'reviews.npy', mode='w+', dtype=np.float32, shape=(n_feature_rows, 384))
        stats[0] = [0., 3., 0., 0., 0., 1., 1.]
        reviews[0] = 0.
        prefix_features(events, active_anchors, vectors, stats, reviews)
        stats.flush(); reviews.flush()
        del vectors
        gc.collect()
        # Intermediate vector lookup is reproducible from Input and not part of the model Output.
        (vector_root / 'review_vectors.npy').unlink(missing_ok=True)
    finally:
        if temp_context is not None:
            temp_context.cleanup()
    train_rows, query_rows = feature_codes[:len(examples)], feature_codes[len(examples):]
    # Only continuous counts/rates are scaled. Missing indicators remain binary.
    count_columns = [0, 4]
    continuous = [0, 1, 2, 3, 4]
    sums = np.zeros(len(continuous), np.float64); squares = sums.copy()
    for start in range(0, len(train_rows), 8192):
        block = np.asarray(stats[train_rows[start:start + 8192]], np.float64)
        block[:, count_columns] = np.log1p(block[:, count_columns])
        values = block[:, continuous]
        sums += values.sum(axis=0); squares += (values * values).sum(axis=0)
    mean = np.zeros(len(FEATURE_SCHEMA), np.float32); scale = np.ones(len(FEATURE_SCHEMA), np.float32)
    mean[continuous] = sums / len(train_rows)
    scale[continuous] = np.sqrt(np.maximum(squares / len(train_rows) - (sums / len(train_rows)) ** 2, 0))
    scale[scale < 1e-6] = 1
    for name, array in [('train_feature_rows', train_rows), ('query_feature_rows', query_rows),
                        ('scaler_mean', mean), ('scaler_scale', scale)]:
        rec.save_array(root / (name + '.npy'), array)
    empty_anchors = len(unique_anchors) - len(active_anchors)
    state = dict(status='complete', signature=signature, schema=FEATURE_SCHEMA, anchors=len(unique_anchors),
        feature_rows=n_feature_rows, consumed_review_texts=len(wanted),
        events=len(events), events_with_text=int(events.has_text.sum()),
        empty_rating_anchors=int(empty_anchors + (stats[1:, 5] == 1).sum()),
        empty_review_anchors=int(empty_anchors + (stats[1:, 6] == 1).sum()),
        policy='strict-before whole timestamp; event-weighted review mean then L2; scaler fit positive train anchors only',
        files={p.name: rec.sha256(p) for p in root.glob('*.npy')})
    rec.save_json(complete, state)
    del stats, reviews
    return feature_arrays(root)


class Features:
    def __init__(self, variant, item_vectors=None, item_valid=None, d2=None):
        self.variant = {'TT0': 'D0', 'TT1': 'D1', 'TT2': 'D2'}.get(variant, variant)
        self.items, self.valid, self.d2 = item_vectors, item_valid, d2

    def arrays(self, items, rows):
        return dict(self.item_arrays(items), **self.user_arrays(rows))

    def item_arrays(self, items):
        output = {}
        if self.variant in ('D1', 'D2'):
            output['item_text'] = np.array(self.items[items], dtype=np.float32, copy=True)
            output['item_present'] = self.valid[items].astype(np.float32)
        return output

    def user_arrays(self, rows):
        output = {}
        if self.variant == 'D2':
            stats = np.array(self.d2['stats'][rows], dtype=np.float32, copy=True)
            stats[:, [0, 4]] = np.log1p(stats[:, [0, 4]])
            stats = (stats - self.d2['scaler_mean']) / self.d2['scaler_scale']
            output['stats'] = stats
            output['reviews'] = np.array(self.d2['reviews'][rows], copy=True)
        return output

    def tensors(self, items, rows, device):
        return {key: torch.as_tensor(value, device=device) for key, value in self.arrays(items, rows).items()}


class ScoringFeatures:
    """Checkpoint-local projections: no repeated 384-D transfer/projection for every user-item pair."""
    def __init__(self, model, features, feature_rows, device, batch=8192):
        self.variant = features.variant
        self.items = self.present = self.rows = self.stats = self.reviews = None
        if self.variant not in ('D1', 'D2'):
            return
        needed_rows = np.unique(feature_rows) if self.variant == 'D2' else np.empty(0, np.int64)
        estimate = len(features.items) * (model.item_projection.out_features + 1) * 4
        if self.variant == 'D2':
            estimate += len(needed_rows) * (model.review_projection.out_features + len(FEATURE_SCHEMA) + 2) * 4
        if str(device).startswith('cuda:') and estimate > torch.cuda.mem_get_info(device)[0] * .5:
            raise RuntimeError('Scoring feature cache exceeds half free VRAM; set SCORE_FEATURE_CACHE=False explicitly')
        with torch.inference_mode():
            self.items = torch.empty((len(features.items), model.item_projection.out_features), device=device)
            self.present = torch.as_tensor(features.valid.astype(np.float32), device=device)
            for start in range(0, len(features.items), batch):
                stop = min(start + batch, len(features.items))
                text = torch.as_tensor(np.array(features.items[start:stop], copy=True), device=device)
                self.items[start:stop] = torch.relu(model.item_projection(text))
            if self.variant == 'D2':
                self.rows = torch.as_tensor(needed_rows, device=device)
                self.stats = torch.empty((len(needed_rows), len(FEATURE_SCHEMA)), device=device)
                self.reviews = torch.empty((len(needed_rows), model.review_projection.out_features), device=device)
                for start in range(0, len(needed_rows), batch):
                    stop = min(start + batch, len(needed_rows)); rows = needed_rows[start:stop]
                    stats = np.array(features.d2['stats'][rows], dtype=np.float32, copy=True)
                    stats[:, [0, 4]] = np.log1p(stats[:, [0, 4]])
                    self.stats[start:stop] = torch.as_tensor((stats - features.d2['scaler_mean']) / features.d2['scaler_scale'], device=device)
                    reviews = torch.as_tensor(np.array(features.d2['reviews'][rows], copy=True), device=device)
                    self.reviews[start:stop] = torch.relu(model.review_projection(reviews))

    def tensors(self, items, rows, device):
        if self.variant not in ('D1', 'D2'):
            return {}
        items = torch.as_tensor(items, device=device, dtype=torch.long)
        output = dict(item_projected=self.items[items], item_present=self.present[items])
        if self.variant == 'D2':
            positions = torch.searchsorted(self.rows, torch.as_tensor(rows, device=device, dtype=torch.long))
            output.update(stats=self.stats[positions], review_projected=self.reviews[positions])
        return output


def training_batches(examples, binary, features, epoch, config):
    """At most one prepared CPU batch ahead; pinning runs off the GPU consumer thread."""
    pin = str(config['train_device']).startswith('cuda:')
    def prepare():
        iterator = iter(epoch_batches(examples, binary, epoch, config))
        while True:
            started = time.monotonic()
            raw = next(iterator, None)
            if raw is None:
                return
            users, items, labels, anchors = raw
            rows = features.d2['train_feature_rows'][anchors] if features.d2 else anchors
            arrays = dict(users=users, items=items, labels=labels, **features.arrays(items, rows))
            tensors = {key: torch.from_numpy(np.ascontiguousarray(value)) for key, value in arrays.items()}
            if pin:
                tensors = {key: value.pin_memory() for key, value in tensors.items()}
            yield tensors, int(labels.sum()), time.monotonic() - started
    iterator = iter(prepare())
    if not config.get('prefetch_train', True):
        yield from iterator
        return
    with ThreadPoolExecutor(max_workers=1) as executor:
        pending = executor.submit(next, iterator, None)
        while True:
            batch = pending.result()
            if batch is None:
                break
            pending = executor.submit(next, iterator, None)
            yield batch


def atomic_checkpoint(path, state, config):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    # Free space already excludes the old file; reserve the new payload before replacement.
    tensor_bytes = sum(t.numel() * t.element_size() for t in state['model'].values())
    rec.disk_guard(path.parent, tensor_bytes * (3 if 'optimizer' in state else 1) + 1024**2, config['min_free_gib'])
    temporary = path.with_suffix('.pt.tmp')
    torch.save(state, temporary)
    temporary.replace(path)
    rec.save_json(path.with_suffix('.json'), dict(sha256=rec.sha256(path), signature=state['signature'], epoch=state['epoch']))


def read_checkpoint(path, signature, device='cpu'):
    meta = rec.read_json(Path(path).with_suffix('.json'))
    if meta.get('signature') != signature or rec.sha256(path) != meta['sha256']:
        raise ValueError('Model checkpoint source/checksum mismatch')
    state = torch.load(path, map_location=device, weights_only=True)
    if state['signature'] != signature or state['epoch'] != meta['epoch']:
        raise ValueError('Model checkpoint metadata mismatch')
    return state


def score_block_tensor(model, users, items, feature_rows, features, device, pair_batch):
    """Keep pair logits on device until Top-K reduction, instead of copying each microbatch."""
    scores = torch.empty((len(users), len(items)), device=device, dtype=torch.float32)
    width = len(items)
    device_indices = isinstance(features, ScoringFeatures) or features.variant not in ('D1', 'D2')
    if device_indices:
        user_tensor = torch.as_tensor(users, device=device, dtype=torch.long)
        item_tensor = torch.as_tensor(items, device=device, dtype=torch.long)
        row_tensor = torch.as_tensor(feature_rows, device=device, dtype=torch.long)
    with torch.inference_mode():
        for start in range(0, len(users) * width, pair_batch):
            end = min(start + pair_batch, len(users) * width)
            if device_indices:
                flat = torch.arange(start, end, device=device)
                q, i = flat // width, item_tensor[flat % width]
                values = model(user_tensor[q], i, **features.tensors(i, row_tensor[q], device))
            else:
                flat = np.arange(start, end)
                q, i = flat // width, items[flat % width]
                values = model(torch.as_tensor(users[q], device=device), torch.as_tensor(i, device=device),
                               **features.tensors(i, feature_rows[q], device))
            scores.ravel()[start:end] = values
    return scores


def score_block(model, users, items, feature_rows, features, device, pair_batch):
    scores = score_block_tensor(model, users, items, feature_rows, features, device, pair_batch).cpu().numpy()
    if not np.isfinite(scores).all():
        raise ValueError('Neural scorer produced NaN/Inf')
    return scores


def device_topk(scores, ids, k=20):
    """Exact FP32 score-desc/item-ID-asc ordering in one sortable int64 key, including ties."""
    if scores.dtype != torch.float32 or scores.ndim != 2:
        raise ValueError('FP32 matrix required for exact device Top-K')
    ids = torch.broadcast_to(torch.as_tensor(ids, device=scores.device, dtype=torch.int64), scores.shape)
    # Canonicalize +/-0 so numeric ties use only item IDs. IDs are nonnegative Int32 Gold indices.
    bits = torch.where(scores == 0, torch.zeros_like(scores), scores).contiguous().view(torch.int32).to(torch.int64)
    ordered = torch.where(bits < 0, ~bits, bits ^ 0x80000000) & 0xffffffff
    keys = ((ordered - 0x80000000) << 32) | (0xffffffff - ids)
    selected = torch.topk(keys, min(k, scores.shape[1]), dim=1, sorted=True).indices
    return torch.gather(ids, 1, selected), torch.gather(scores, 1, selected)


class TowerScoringFeatures:
    """One item-output matrix per checkpoint/device, shared across scoring shards."""
    def __init__(self, model, features, nitems, device, batch=8192):
        self.features = features
        self.items = torch.empty((nitems, model.output_dim), dtype=torch.float32, device=device)
        with torch.inference_mode():
            for start in range(0, nitems, batch):
                items = np.arange(start, min(start + batch, nitems), dtype=np.int64)
                inputs = {key: torch.as_tensor(value, device=device) for key, value in features.item_arrays(items).items()}
                self.items[start:start + len(items)] = model.encode_item(torch.as_tensor(items, device=device), **inputs)
        if not bool(torch.isfinite(self.items).all()):
            raise ValueError('Non-finite item tower output')


def rank_tower(model, users, feature_rows, cache, config, device):
    output = np.full((len(users), 20), -1, np.int32)
    values = np.full((len(users), 20), -np.inf, np.float32)
    with torch.inference_mode():
        for start in range(0, len(users), config['query_batch']):
            stop = min(start + config['query_batch'], len(users))
            inputs = {key: torch.as_tensor(value, device=device)
                      for key, value in cache.features.user_arrays(feature_rows[start:stop]).items()}
            query = model.encode_user(torch.as_tensor(users[start:stop], device=device), **inputs)
            best_ids = torch.empty((stop - start, 0), device=device, dtype=torch.long)
            best_scores = torch.empty((stop - start, 0), device=device)
            finite = torch.isfinite(query).all()
            for lo in range(0, len(cache.items), config['item_block']):
                items = cache.items[lo:lo + config['item_block']]
                scores = query @ items.T
                finite &= torch.isfinite(scores).all()
                ids, scores = device_topk(scores, np.arange(lo, lo + len(items)))
                best_ids, best_scores = device_topk(torch.cat([best_scores, scores], dim=1), torch.cat([best_ids, ids], dim=1))
            if not bool(finite):
                raise ValueError('Non-finite Two-Tower scores')
            width = best_ids.shape[1]
            output[start:stop, :width] = best_ids.cpu().numpy()
            values[start:stop, :width] = best_scores.cpu().numpy()
    return output, values


def rank_neural(model, users, feature_rows, features, nitems, config, device):
    model.eval()
    if isinstance(model, TwoTowerNetwork):
        cache = features if isinstance(features, TowerScoringFeatures) else TowerScoringFeatures(model, features, nitems, device)
        return rank_tower(model, users, feature_rows, cache, config, device)
    if config.get('score_feature_cache', True) and not isinstance(features, ScoringFeatures):
        features = ScoringFeatures(model, features, feature_rows, device)
    output = np.full((len(users), 20), -1, np.int32)
    values = np.full((len(users), 20), -np.inf, np.float32)
    for start in range(0, len(users), config['query_batch']):
        end = min(len(users), start + config['query_batch'])
        best_ids = torch.empty((end - start, 0), device=device, dtype=torch.long)
        best_scores = torch.empty((end - start, 0), device=device, dtype=torch.float32)
        finite = torch.tensor(True, device=device)
        for lo in range(0, nitems, config['item_block']):
            items = np.arange(lo, min(nitems, lo + config['item_block']), dtype=np.int64)
            scores = score_block_tensor(model, users[start:end], items, feature_rows[start:end], features, device, config['pair_batch'])
            finite &= torch.isfinite(scores).all()
            ids, score = device_topk(scores, items)
            best_ids, best_scores = device_topk(torch.cat([best_scores, score], dim=1), torch.cat([best_ids, ids], dim=1))
        if not bool(finite):
            raise ValueError('Neural scorer produced NaN/Inf')
        best_ids, best_scores = best_ids.cpu().numpy(), best_scores.cpu().numpy()
        output[start:end, :best_ids.shape[1]] = best_ids
        values[start:end, :best_scores.shape[1]] = best_scores
    return output, values


def predictions(model, fold, features, root, signature, config, only_warm=False):
    root = Path(root); root.mkdir(parents=True, exist_ok=True)
    positions = np.flatnonzero((fold.user_indices >= 0) & (fold.warm if only_warm else True))
    output = np.full((fold.nqueries, 20), -1, np.int32)
    feature_rows = features.d2['query_feature_rows'] if features.d2 else np.zeros(fold.nqueries, np.int64)
    digest = rec.fingerprint(dict(signature=signature, positions=rec.array_hash(positions), nitems=fold.nitems,
                                  dtype='float32', devices=config['score_devices'],
                                  feature_cache=config.get('score_feature_cache', True),
                                  gpu_topk='float32_int64_lexicographic_v1'))
    shards = [(number, start, min(len(positions), start + config['score_shard_queries']))
              for number, start in enumerate(range(0, len(positions), config['score_shard_queries']))]
    # CPU template avoids retaining a second full train model/optimizer on a scoring GPU.
    devices = config['score_devices']
    def worker(rank):
        device = devices[rank]
        local = local_features = None
        try:
            for number, start, end in shards[rank::len(devices)]:
                query_positions = positions[start:end]
                path = root / f'part-{number:06d}.npz'
                meta_path = path.with_suffix('.json')
                if path.exists() and meta_path.exists():
                    meta = rec.read_json(meta_path)
                    if meta.get('signature') != digest or rec.sha256(path) != meta['sha256']:
                        raise ValueError('Neural scoring checkpoint mismatch')
                    with np.load(path, allow_pickle=False) as saved:
                        if not np.array_equal(saved['query_positions'], query_positions) or saved['items'].shape != (len(query_positions), 20):
                            raise ValueError('Wrong scoring query shard')
                        selected = saved['items'].copy()
                else:
                    if local is None:
                        local = copy.deepcopy(model).to(device).eval()
                        if isinstance(local, TwoTowerNetwork):
                            local_features = TowerScoringFeatures(local, features, fold.nitems, device)
                        else:
                            local_features = (ScoringFeatures(local, features, feature_rows[positions], device)
                                if config.get('score_feature_cache', True) else features)
                    started = time.monotonic()
                    selected, scores = rank_neural(local, fold.user_indices[query_positions].astype(np.int64),
                        feature_rows[query_positions], local_features, fold.nitems, config, device)
                    rec.disk_guard(root, selected.nbytes + scores.nbytes, config['min_free_gib'])
                    temporary = path.with_suffix('.npz.tmp')
                    with temporary.open('wb') as stream:
                        np.savez(stream, query_positions=query_positions, items=selected, scores=scores)
                    temporary.replace(path)
                    seconds = time.monotonic() - started
                    rec.save_json(meta_path, dict(signature=digest, sha256=rec.sha256(path), queries=len(query_positions),
                        seconds=seconds, pairs_per_second=len(query_positions) * fold.nitems / max(seconds, 1e-9), device=device))
                output[query_positions] = selected
                print('Scored/reused', fold.name, end, '/', len(positions), 'queries', flush=True)
        finally:
            del local, local_features
    with ThreadPoolExecutor(max_workers=len(devices)) as executor:
        list(executor.map(worker, range(len(devices))))
    rec.save_json(root / 'manifest.json', dict(status='complete', signature=digest, queries=len(positions), shards=len(shards)))
    return output


def checkpoint_state(model, optimizer, epoch, signature, trial_state):
    # Tensor/primitive-only format permits weights_only=True on all restore paths.
    return dict(signature=signature, epoch=epoch, model=model.state_dict(), optimizer=optimizer.state_dict(),
                torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                trial_state=copy.deepcopy(trial_state))


def training_plan(config):
    plan = dict({key: config[key] for key in TRAINING_SIGNATURE_KEYS}, negative_sampler=SAMPLER_VERSION)
    if config['algorithm'] == 'two_tower':
        plan.update(tower_output_dim=config['tower_output_dim'], architecture='independent_mlp_dot_v1', loss='bce_mean')
    return plan


def early_stop_state(config):
    return dict(enabled=config['early_stopping'], monitor='warm_full_ndcg_at10',
        start_epoch=config['early_stop_start_epoch'], patience=config['early_stop_patience'],
        min_delta=float(config['early_stop_min_delta']), significant_best=None,
        bad_checks=0, last_checked_epoch=None, last_metric=None, stop_requested=False)


def update_early_stopping(control, epoch, ndcg):
    """Count completed validation checks, independently of strict best-checkpoint selection."""
    if not np.isfinite(ndcg):
        raise ValueError('Non-finite early-stopping metric')
    if control['last_checked_epoch'] is not None and epoch <= control['last_checked_epoch']:
        if epoch == control['last_checked_epoch'] and ndcg == control['last_metric']:
            return  # Idempotent after a completed-check resume.
        raise ValueError('Early-stopping checks must advance in epoch order')
    if control['stop_requested']:
        raise ValueError('Cannot update a stopped trial with a later validation')
    reference = control['significant_best']
    improved = reference is None or (ndcg > reference and ndcg >= reference + control['min_delta'])
    if improved:
        control['significant_best'] = float(ndcg)
        control['bad_checks'] = 0
    elif not control['enabled'] or epoch < control['start_epoch']:
        control['bad_checks'] = 0
    else:
        control['bad_checks'] += 1
    control.update(last_checked_epoch=epoch, last_metric=float(ndcg))
    control['stop_requested'] = bool(control['enabled'] and epoch >= control['start_epoch']
                                     and control['bad_checks'] >= control['patience'])


def train_epoch(model, optimizer, fold, examples, features, epoch, config):
    model.train()
    total = positive = 0
    loss_fn = nn.BCEWithLogitsLoss()
    device = config['train_device']
    cuda = str(device).startswith('cuda:')
    if cuda:
        torch.cuda.synchronize(device)
    started = time.monotonic()
    loss_sum = torch.zeros((), device=device)
    prepare_seconds = wait_seconds = step_seconds = 0.
    batch_number = 0
    with closing(training_batches(examples, fold.binary, features, epoch, config)) as batches:
        while True:
            waiting = time.monotonic()
            batch = next(batches, None)
            wait_seconds += time.monotonic() - waiting
            if batch is None:
                break
            batch_number += 1
            tensors, positives, preparation = batch
            prepare_seconds += preparation
            stepping = time.monotonic()
            tensors = {key: value.to(device, non_blocking=cuda) for key, value in tensors.items()}
            users, items, labels = (tensors.pop(key) for key in ('users', 'items', 'labels'))
            optimizer.zero_grad(set_to_none=True)
            logits = model(users, items, **tensors)
            loss = loss_fn(logits, labels)
            loss.backward()
            # One host synchronization before update, instead of one per parameter plus loss.item().
            finite = [torch.isfinite(loss)] + [torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None]
            if not bool(torch.stack(finite).all()):
                raise ValueError('Non-finite train loss or neural gradient')
            optimizer.step()
            loss_sum += loss.detach() * len(labels)
            total += len(labels); positive += positives
            step_seconds += time.monotonic() - stepping
            if batch_number == 1 or batch_number % 100 == 0:
                print('Train epoch', epoch, 'positives', positive, '/', len(examples), 'BCE', round(float(loss_sum) / total, 6), flush=True)
    if positive != len(examples):
        raise ValueError('Training dropped positive anchors')
    if cuda:
        torch.cuda.synchronize(device)
    elapsed = time.monotonic() - started
    return dict(loss=float(loss_sum) / total, pairs=total, positives=positive, negatives=total - positive,
        optimizer_steps=batch_number, seconds=elapsed, pairs_per_second=total / max(elapsed, 1e-9),
        sample_prepare_cpu_seconds=prepare_seconds, waiting_for_batch_seconds=wait_seconds,
        step_host_seconds=step_seconds,
        timing_policy='CPU prepare overlaps GPU; step_host includes enqueue/finite-check waits, not isolated GPU kernel time')


def parameter_budget(nusers, nitems, variant, parameters, config):
    dim = parameters['dim']
    count = (nusers + nitems) * dim * (2 if variant == 'NCF_ID' else 1)
    width = 2 * dim
    if variant != 'NCF_ID':
        count += nusers + nitems + 1
    if variant in ('D1', 'D2'):
        count += 385 * config['projection_dim']; width += config['projection_dim'] + 1
    if variant == 'D2':
        count += 385 * config['projection_dim']; width += config['projection_dim'] + len(FEATURE_SCHEMA)
    for hidden in parameters['hidden']:
        count += (width + 1) * hidden; width = hidden
    count += width + (dim if variant == 'NCF_ID' else 0) + 1
    if variant in TOWER_VARIANTS:
        count = (nusers + nitems) * dim
        widths = [dim, dim]
        if variant in ('TT1', 'TT2'):
            count += 385 * config['projection_dim']
            widths[1] += config['projection_dim'] + 1
        if variant == 'TT2':
            count += 385 * config['projection_dim']
            widths[0] += config['projection_dim'] + len(FEATURE_SCHEMA)
        for width in widths:
            for hidden in parameters['hidden'] + [config['tower_output_dim']]:
                count += (width + 1) * hidden
                width = hidden
    import psutil
    free = psutil.virtual_memory().available if config['train_device'] == 'cpu' else torch.cuda.mem_get_info(config['train_device'])[0]
    minimum = count * 16
    estimated = minimum + (count * 4 if config['adam_foreach'] else 0)
    print('Parameter budget:', count, '; minimum FP32 Adam bytes:', estimated, '; available:', free, flush=True)
    if estimated > free * .8:
        raise RuntimeError('Model/Adam budget exceeds 80% free memory before activations; choose a smaller locked trial dimension/device.')
    return dict(parameters=count, minimum_adam_bytes=minimum, estimated_train_bytes=estimated,
                adam_foreach=config['adam_foreach'], available_bytes=free)


def fit_trial(fold, examples, features, variant, parameters, root, source, config, epochs=None):
    root.mkdir(parents=True, exist_ok=True)
    if epochs is not None and (type(epochs) is not int or not 1 <= epochs <= config['max_epochs']):
        raise ValueError('Refit epochs must be a locked validation selection within the training budget')
    signature = rec.fingerprint(dict(source=source, fold=fold.name, parameters=parameters, variant=variant,
                                     refit_epochs=epochs, training_plan=training_plan(config)))
    metadata = root / 'trial.json'
    state = rec.read_json(metadata) if metadata.exists() else dict(signature=signature, status='running',
        epochs=[], evaluations=[], best=None, early_stopping=early_stop_state(config))
    if state['signature'] != signature:
        raise ValueError('Trial metadata mismatch')
    if state.get('status') == 'complete' and state.get('model_pruned'):
        return state, signature
    last = root / 'last.pt'
    restored = read_checkpoint(last, signature) if last.exists() else None
    if restored is not None:
        # The atomic model checkpoint commits epoch/evaluation/control together. The JSON
        # mirror can lag if the process stopped after committing the checkpoint.
        state = restored.get('trial_state')
        if not state or state.get('signature') != signature or 'early_stopping' not in state:
            raise ValueError('Checkpoint lacks matching compact training state')
    if state.get('status') == 'complete':
        if restored is None or restored['epoch'] != state['stopped_epoch']:
            raise ValueError('Completed trial missing its matching final checkpoint')
        rec.save_json(metadata, state)
        return state, signature
    budget = parameter_budget(fold.nusers, fold.nitems, variant, parameters, config)
    seed_model(config['seed'])
    model = make_model(fold.nusers, fold.nitems, variant, parameters, config).to(config['train_device'])
    if sum(p.numel() for p in model.parameters()) != budget['parameters']:
        raise AssertionError('Parameter estimator out of sync')
    optimizer = torch.optim.Adam(model.parameters(), lr=parameters['lr'], weight_decay=parameters['weight_decay'],
                                 foreach=config['adam_foreach'])
    completed = 0
    if restored is not None:
        model.load_state_dict(restored['model']); optimizer.load_state_dict(restored['optimizer'])
        completed = restored['epoch']
        if not state['epochs'] or state['epochs'][-1]['epoch'] != completed:
            raise ValueError('Checkpoint epoch and loss history disagree')
        torch.set_rng_state(restored['torch_rng'])
        if restored['cuda_rng']:
            torch.cuda.set_rng_state_all(restored['cuda_rng'])
        del restored
    limit = epochs if epochs is not None else config['max_epochs']
    if completed > limit:
        raise ValueError('Checkpoint exceeds requested epoch budget')
    for epoch in range(max(1, completed), limit + 1):
        if epochs is None and state['early_stopping']['stop_requested']:
            break
        if epoch > completed:
            with rec.Monitor(list(set(config['score_devices'] + [config['train_device']]))) as monitor:
                measures = train_epoch(model, optimizer, fold, examples, features, epoch, config)
            state['epochs'] = [e for e in state['epochs'] if e['epoch'] != epoch]
            state['epochs'].append(dict(epoch=epoch, **measures, resources=monitor.result))
            atomic_checkpoint(last, checkpoint_state(model, optimizer, epoch, signature, state), config)
            completed = epoch
            rec.save_json(metadata, state)
            print(variant, parameters['name'], 'epoch', epoch, measures, flush=True)
        if epochs is None and epoch in config['eval_epochs'] and not any(e['epoch'] == epoch for e in state['evaluations']):
            model.zero_grad(set_to_none=True)
            model.cpu()
            with rec.Monitor(config['score_devices']) as scoring:
                native = predictions(model, fold, features, root / f'epoch-{epoch}-ranking',
                    dict(trial=signature, model_sha256=rec.sha256(last)), config, only_warm=True)
            measures = rec.query_metrics(fold, native)
            if not fold.warm.any():
                raise ValueError('No warm-full validation queries')
            result = dict(epoch=epoch, trial=parameters['name'], parameters=parameters,
                ndcg_at10=float(measures['ndcg@10'][fold.warm].mean()), recall_at10=float(measures['recall@10'][fold.warm].mean()),
                parameter_count=budget['parameters'], scoring_resources=scoring.result)
            if state['best'] is None or selection_score(result) > selection_score(state['best']):
                # Best is inference-only; the last checkpoint retains optimizer/RNG for epoch resume.
                atomic_checkpoint(root / 'best.pt', dict(signature=signature, epoch=epoch, model=model.state_dict()), config)
                state['best'] = result
            update_early_stopping(state['early_stopping'], epoch, result['ndcg_at10'])
            result['bad_checks'] = state['early_stopping']['bad_checks']
            result['significant_best_ndcg'] = state['early_stopping']['significant_best']
            state['evaluations'].append(result)
            # Commit before updating the JSON mirror so resume never double-counts patience.
            atomic_checkpoint(last, checkpoint_state(model, optimizer, epoch, signature, state), config)
            rec.save_json(metadata, state)
            print(variant, parameters['name'], 'validation epoch', epoch, 'NDCG@10', result['ndcg_at10'],
                  'bad checks', state['early_stopping']['bad_checks'], '/', config['early_stop_patience'], flush=True)
            if state['early_stopping']['stop_requested']:
                break
            model.to(config['train_device'])
    if not completed or (epochs is None and state['best'] is None):
        raise ValueError('Trial completed without training or validation selection')
    rec.save_json(root / 'fit_budget.json', budget)
    pd.DataFrame([{k: v for k, v in e.items() if k != 'resources'} for e in state['epochs']]).to_csv(root / 'loss.csv', index=False)
    if state['evaluations']:
        pd.DataFrame([{k: v for k, v in e.items() if k not in ('parameters', 'scoring_resources')}
                      for e in state['evaluations']]).to_csv(root / 'validation.csv', index=False)
    reason = ('refit_selected_epoch' if epochs is not None else
              'ranking_plateau' if state['early_stopping']['stop_requested'] else 'max_epochs')
    recent = state['evaluations'][-2:]
    improving_at_cap = bool(reason == 'max_epochs' and len(recent) == 2
                            and recent[-1]['ndcg_at10'] > recent[-2]['ndcg_at10'])
    state.update(status='complete', optimizer_retained=False, stop_reason=reason, stopped_epoch=completed,
                 selected_epoch=epochs if epochs is not None else state['best']['epoch'],
                 budget_exhausted_while_improving=improving_at_cap)
    # A finished, locked trial/refit never trains further. Release Adam moments before upload;
    # interrupted epochs retain full optimizer/RNG in last.pt until this completion boundary.
    model.zero_grad(set_to_none=True)
    model.cpu()
    atomic_checkpoint(last, dict(signature=signature, epoch=completed, model=model.state_dict(),
                                 trial_state=copy.deepcopy(state)), config)
    rec.save_json(metadata, state)
    print(variant, parameters['name'], 'finished:', reason, '; trained epoch', completed,
          '; selected epoch', state['selected_epoch'], flush=True)
    if improving_at_cap:
        print('Epoch budget exhausted while validation is still improving; convergence is not established.', flush=True)
    del model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return state, signature


def prune_losing_trials(root, results):
    """Delete only generated model payloads of completed, strictly non-selected trials."""
    winner = max(results, key=selection_score)['trial']
    trials_root = (root / 'trials').resolve()
    for result in results:
        if result['trial'] == winner:
            continue
        directory = rec.child(trials_root, result['trial'])
        state = rec.read_json(directory / 'trial.json')
        if state.get('status') != 'complete':
            raise ValueError('Cannot prune an incomplete trial')
        for name in ('last.pt', 'last.json', 'best.pt', 'best.json'):
            (directory / name).unlink(missing_ok=True)
        state['model_pruned'] = True
        rec.save_json(directory / 'trial.json', state)


def compact_d2(root, config, queries):
    """Retain query features/scaler for inspection; training prefix arrays are transient after scoring."""
    state = rec.read_json(root / 'features.json')
    rec.save_json(root / 'features.json', dict(state, status='compacting'))
    rows = np.load(root / 'query_feature_rows.npy')
    rec.disk_guard(root, len(rows) * (384 + len(FEATURE_SCHEMA)) * 4, config['min_free_gib'])
    for name in ('stats', 'reviews'):
        original = np.load(root / (name + '.npy'), mmap_mode='r')
        target = np.lib.format.open_memmap(root / ('query_' + name + '.npy'), mode='w+',
                                           dtype=np.float32, shape=(len(rows), original.shape[1]))
        for start in range(0, len(rows), 8192):
            target[start:start + 8192] = original[rows[start:start + 8192]]
        target.flush()
        del original, target
    gc.collect()
    for name in ('stats.npy', 'reviews.npy', 'train_feature_rows.npy', 'query_feature_rows.npy'):
        (root / name).unlink()
    state['cache_compacted'] = True
    state['status'] = 'complete'
    queries[['query_id', 'user_idx', 'prediction_time']].to_parquet(root / 'query_keys.parquet', index=False)
    state['files'] = {p.name: rec.sha256(p) for p in root.iterdir() if p.suffix in ('.npy', '.parquet')}
    rec.save_json(root / 'features.json', state)


def selection_score(result):
    return (result['ndcg_at10'], result['recall_at10'], -result['parameter_count'], -result['epoch'])


def phase_valid(root, phase, source):
    path = root / phase / 'phase_complete.json'
    if not path.exists():
        return False
    state = rec.read_json(path)
    return (state.get('status') == 'complete' and state.get('source') == source and bool(state.get('artifacts'))
            and all(rec.child(root, p).is_file() and rec.sha256(rec.child(root, p)) == digest for p, digest in state['artifacts'].items()))


def run(config):
    validate_config(config)
    phase, variant = config['phase'], config['variant']
    if phase not in rec.BOUNDS or variant not in config['variants'] or config['category'] not in config['categories']:
        raise ValueError('Invalid phase/variant/category')
    gold = rec.Gold(rec.discover_gold(config.get('gold_root')), config['category'])
    embedding = rec.discover_embedding(gold, config.get('embedding_root'), config.get('embedding_bundle')) if variant in ('D1', 'D2', 'TT1', 'TT2') else None
    source = dict(gold=gold.signature, variant=variant, algorithm=config['algorithm'], protocol=rec.PROTOCOL,
        shared_code=rec.sha256(rec.__file__), neural_code=rec.sha256(__file__), versions=rec.versions(),
        embedding=rec.sha256(embedding) if embedding else None,
        training=training_plan(config),
        dataset_versions=config.get('dataset_versions'))
    parent = Path(config['output_root']).resolve(); parent.mkdir(parents=True, exist_ok=True)
    root = parent / (variant.lower() + '_' + config['category'] + '_' + rec.fingerprint(source)[:16])
    rec.resume_output(root, Path(config['resume_root']) / root.name if config.get('resume_root') else None, config['min_free_gib'])
    root.mkdir(parents=True, exist_ok=True)
    if (root / 'run.json').exists() and rec.read_json(root / 'run.json')['signature'] != source:
        raise ValueError('Neural run source mismatch')
    if phase == 'test' and not phase_valid(root, 'validation', source):
        raise ValueError('Run/restore matching complete validation first; test cannot select configuration')
    if phase_valid(root, phase, source):
        print('Reused complete neural phase:', variant, config['category'], phase, flush=True)
        return root
    phase_root = root / phase; phase_root.mkdir(exist_ok=True)
    rec.save_json(root / 'run.json', dict(status='running', signature=source, phase=phase, config=config))
    try:
        devices = list(set(config['score_devices'] + [config['train_device']]))
        rec.save_json(phase_root / 'runtime.json', dict(python=os.sys.version, torch=torch.__version__,
            torch_cuda=torch.version.cuda, cpu_threads=config['cpu_threads'], train_device=config['train_device'],
            score_devices=config['score_devices'], dtype='float32', deterministic_algorithms=True,
            gpus=[dict(device=d, name=torch.cuda.get_device_name(d), total_memory=torch.cuda.get_device_properties(d).total_memory)
                  for d in devices if d.startswith('cuda:')]))
        with rec.Monitor(devices) as loading:
            fold = gold.fold(phase)
            examples = load_examples(gold, fold)
        rec.save_json(phase_root / 'load_resources.json', loading.result)
        fold.copy_mappings(phase_root)
        with rec.Monitor(devices) as preparing:
            item_vectors = item_valid = d2 = None
            if embedding:
                item_vectors, item_valid, info = rec.load_item_vectors(gold, fold, embedding.parent, embedding.parent.name)
                rec.disk_guard(root, item_vectors.nbytes, config['min_free_gib'])
                rec.save_array(phase_root / 'item_vectors.npy', item_vectors)
                rec.save_array(phase_root / 'item_valid.npy', item_valid)
                rec.save_json(phase_root / 'item_features.json', info)
            if variant in ('D2', 'TT2'):
                d2 = build_d2(gold, fold, examples, embedding, phase_root / 'features', config)
            features = Features(variant, item_vectors, item_valid, d2)
        rec.save_json(phase_root / 'feature_resources.json', preparing.result)
        best_path = root / 'best_config.json'
        if phase == 'validation':
            results = []
            for parameters in config['trials']:
                trial_root = root / 'trials' / parameters['name']
                trial, trial_signature = fit_trial(fold, examples, features, variant, parameters, trial_root, source, config)
                results.append(dict(trial['best'], stopped_epoch=trial['stopped_epoch'], selected_epoch=trial['selected_epoch'],
                                    stop_reason=trial['stop_reason'],
                                    checkpoint=str((trial_root / 'best.pt').relative_to(root).as_posix()),
                                    checkpoint_signature=trial_signature))
                pd.DataFrame([{**r, 'parameters': json.dumps(r['parameters'], sort_keys=True)} for r in results]).to_csv(root / 'trials.csv', index=False)
                prune_losing_trials(root, results)
            best = max(results, key=selection_score)
            best['checkpoint_sha256'] = rec.sha256(root / best['checkpoint'])
            rec.save_json(best_path, dict(status='locked', source=source, best=best))
        else:
            selection = rec.read_json(best_path)
            if selection.get('status') != 'locked' or selection.get('source') != source:
                raise ValueError('No matching locked validation configuration')
            best = selection['best']
            if rec.sha256(root / best['checkpoint']) != best['checkpoint_sha256']:
                raise ValueError('Best validation model changed')
            _, refit_signature = fit_trial(fold, examples, features, variant, best['parameters'],
                phase_root / 'model', source, config, epochs=best['epoch'])
        model_path = root / best['checkpoint'] if phase == 'validation' else phase_root / 'model/last.pt'
        checkpoint_signature = best['checkpoint_signature'] if phase == 'validation' else refit_signature
        restored = read_checkpoint(model_path, checkpoint_signature)
        model = make_model(fold.nusers, fold.nitems, variant, best['parameters'], config)
        model.load_state_dict(restored['model']); del restored
        with rec.Monitor(config['score_devices']) as scoring:
            native = predictions(model, fold, features, phase_root / 'ranking',
                dict(source=source, model=rec.sha256(model_path)), config)
        rec.save_json(phase_root / 'score_resources.json', scoring.result)
        if isinstance(model, TwoTowerNetwork):
            with rec.Monitor(['cpu']) as exporting:
                rec.disk_guard(phase_root, fold.nitems * model.output_dim * 4, config['min_free_gib'])
                cache = TowerScoringFeatures(model.eval(), features, fold.nitems, 'cpu')
                path = phase_root / 'item_output_vectors.npy'
                rec.save_array(path, cache.items.numpy())
                rec.save_json(phase_root / 'tower_export.json', dict(status='complete',
                    model_sha256=rec.sha256(model_path), vector_sha256=rec.sha256(path),
                    rows=fold.nitems, dimension=model.output_dim, order='fold item_idx',
                    score='raw dot product; no L2 output normalization'))
                del cache
            rec.save_json(phase_root / 'tower_export_resources.json', exporting.result)
        with rec.Monitor(['cpu']) as evaluating:
            rows = rec.evaluate(fold, native, phase_root / 'evaluation', variant.lower(), config)
        rec.save_json(phase_root / 'metric_resources.json', evaluating.result)
        del model, features, d2
        gc.collect()
        if variant in ('D2', 'TT2'):
            compact_d2(phase_root / 'features', config, fold.queries)
        artifacts = {p.relative_to(root).as_posix(): rec.sha256(p) for p in phase_root.rglob('*')
                     if p.is_file() and p.name != 'phase_complete.json'}
        if phase == 'validation':
            for p in [best_path, root / best['checkpoint'], (root / best['checkpoint']).with_suffix('.json')]:
                artifacts[p.relative_to(root).as_posix()] = rec.sha256(p)
            rec.save_json(root / 'validation_complete.json', dict(status='complete', source=source))
        rec.save_json(phase_root / 'phase_complete.json', dict(status='complete', source=source, fold=phase, artifacts=artifacts))
        rec.save_json(root / 'run.json', dict(status='complete', signature=source, phase=phase, config=config))
        print(pd.DataFrame(rows).query("cohort in ['all_queries','warm_full']").to_string(index=False), flush=True)
        return root
    except BaseException as error:
        rec.save_json(root / 'run.json', dict(status='failed', signature=source, phase=phase, config=config,
                                            error_type=type(error).__name__, error=str(error)))
        raise


def run_suite(config):
    validate_config(config)
    publish = config.get('publish_results', True)
    if publish and config['phases'] != ['validation', 'test']:
        raise ValueError('Private Dataset publishing requires both phases; use Save Version/Output for validation-only runs')
    jobs = [(category, variant) for category in config['categories'] for variant in config['variants']]
    handles, remote = config.get('result_datasets', {}), {}
    if publish:
        selected = [handles.get(variant + '/' + category) for category, variant in jobs]
        if any(not h for h in selected) or len(set(selected)) != len(jobs):
            raise ValueError('Use separate result Dataset handles keyed by VARIANT/Category')
        rec.configure_kaggle_auth()
        remote = {handle: rec.result_dataset_private(handle) for handle in selected}
    # Fail missing inputs before a long first training run.
    for category in config['categories']:
        gold = rec.Gold(rec.discover_gold(config.get('gold_root')), category)
        for variant in config['variants']:
            needed = ['validation/train_examples', 'test/train_examples']
            if variant in ('D1', 'D2', 'TT1', 'TT2'):
                rec.discover_embedding(gold, config.get('embedding_root'), config.get('embedding_bundle'))
            if variant in ('D2', 'TT2'):
                needed += ['review_text_refs', 'history_ratings'] + [phase + '/history_events' for phase in config['phases']]
            missing = [table for table in needed if table not in gold.info['tables']]
            if missing:
                raise ValueError('Missing Gold tables: ' + ', '.join(missing))
    output = Path(config['output_root']); output.mkdir(parents=True, exist_ok=True)
    summary, prepared = [], []
    for category, variant in jobs:
        current = dict(config, category=category, variant=variant)
        handle = handles.get(variant + '/' + category)
        previous = None
        if publish and remote[handle] and config.get('auto_resume', True) and not current.get('resume_root'):
            import kagglehub
            restored = Path(kagglehub.dataset_download(handle))
            previous = rec.read_json(restored / 'result_bundle.json')
            if previous.get('format') != 'gold_recommendation_results_v1' or previous.get('handle') != handle:
                raise ValueError('Unexpected neural result Dataset layout')
            current['resume_root'] = str(restored / 'recommendation_results')
        prepared.append((category, variant, current, handle, previous))
    # Lock every selected Two-Tower validation before exposing any test scores.
    if config['algorithm'] == 'two_tower' and config['phases'] == ['validation', 'test']:
        for category, variant, current, handle, previous in prepared:
            run(dict(current, phase='validation'))
    for category, variant, current, handle, previous in prepared:
        for phase in config['phases']:
            print('\n===', category, variant, phase, '===', flush=True)
            root = run(dict(current, phase=phase))
        receipt = rec.publish_results(root, handle, config['min_free_gib'], previous) if publish else None
        summary.append(dict(category=category, variant=variant, run_root=str(root), dataset=handle, upload=receipt))
        rec.save_json(output / (config['algorithm'] + '_suite_results.json'), summary)
    return summary


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--config', required=True)
    args = parser.parse_args()
    run_suite(rec.read_json(args.config))


## Cấu hình và thứ tự chạy

TRAIN_BATCH là ngân sách scored pairs, không phải số positives: 20480 ≈4096 positives với 4 negatives.
Một GPU train; hai GPU chia query shards khi scoring, không gộp VRAM. Nếu một GPU, dùng
SCORE_DEVICES=['cuda:0']; chạy CPU phải đổi cả TRAIN_DEVICE và SCORE_DEVICES=['cpu'].
Không đủ VRAM thì giảm TRAIN_BATCH/QUERY_BATCH/ITEM_BLOCK; đổi train batch cần chọn lại epoch.
TT2 dùng memmap nhưng vẫn cần nhiều GB disk cho profiles, nhất là hai ngành lớn.

Chạy hai lượt để khóa model trước khi xem test. Lượt 1 dùng RUN_MODE='validation': validation
cho đủ category/variant, không upload; nếu đã có validation run, gắn Output cũ và đặt RESUME_ROOT
để resume/reuse, sau đó Save Version/Output để lưu đủ best checkpoints.
Lượt 2 dùng RUN_MODE='category_test', chỉ một TEST_CATEGORY mỗi lần; gắn Output validation,
đặt RESUME_ROOT đến thư mục `two_tower_results` trong Input, rồi bật upload.
Suite reuse validation, chạy test cho ba T của category đó và upload riêng từng T/category.
Mỗi Dataset upload cần chỗ trống tạm khoảng 2× bundle + 5 GiB. Receipt `submitted_processing`
chỉ xác nhận đã gửi upload; giữ bản local cho tới khi xác minh Dataset đã xử lý và đọc được.
Trước khi dọn run lớn, cell phân tích lưu metrics, per-query và CI thành gói gọn; Save Version/Output
gói này riêng. Không xóa thư mục khi remote chưa được xác minh.


In [ ]:
RUN_MODE = 'validation'  # lượt 1: validation; lượt 2: category_test
VALIDATION_CATEGORIES = ['All_Beauty', 'Video_Games', 'Industrial_and_Scientific']
TEST_CATEGORY = 'All_Beauty'  # lượt test chỉ chọn một category
CATEGORIES_TO_RUN = VALIDATION_CATEGORIES if RUN_MODE == 'validation' else [TEST_CATEGORY]
VARIANTS_TO_RUN = ['TT0', 'TT1', 'TT2']
PHASES = ['validation'] if RUN_MODE == 'validation' else ['validation', 'test']
TRAIN_DEVICE = 'cuda:0'
SCORE_DEVICES = ['cuda:0', 'cuda:1']
GOLD_ROOT = None
EMBEDDING_ROOT = None
BUNDLE_NAME = None
RESUME_ROOT = None  # gắn Output cũ ở cả hai lượt để reuse; lượt test cần Output validation
PUBLISH_RESULTS = RUN_MODE == 'category_test'
RESULT_OWNER = 'thoandanh'
INPUT_VERSIONS = {'gold': None, 'embedding': None}
TRAIN_BATCH = 20480  # scored pairs; ~4096 positive anchors với 4 negatives
QUERY_BATCH = 512
ITEM_BLOCK = 16384
SCORE_SHARD_QUERIES = 2048
ID_EMBED_DIM = 8
MLP_HIDDEN = [64]
PROJECTION_DIM = 16
TOWER_OUTPUT_DIM = 32
NEGATIVES_PER_POSITIVE = 4
MAX_EPOCHS = 20
EVAL_EPOCHS = [1, 3, 6, 9, 12, 15, 18, 20]
EARLY_STOPPING = True
EARLY_STOP_START_EPOCH = 6
EARLY_STOP_PATIENCE = 2
EARLY_STOP_MIN_DELTA = 0.00001
CONFIG = {'algorithm': 'two_tower', 'training_profile': 'compact_full_users', 'variants': ['TT0', 'TT1', 'TT2'], 'categories': ['All_Beauty', 'Video_Games', 'Industrial_and_Scientific'], 'gold_root': None, 'embedding_root': None, 'embedding_bundle': None, 'output_root': '/kaggle/working/two_tower_results', 'resume_root': None, 'seed': 42, 'cpu_threads': 2, 'train_device': 'cuda:0', 'score_devices': ['cuda:0', 'cuda:1'], 'train_batch': 20480, 'query_batch': 512, 'item_block': 16384, 'pair_batch': 65536, 'score_shard_queries': 2048, 'adam_foreach': True, 'prefetch_train': True, 'score_feature_cache': True, 'max_epochs': 20, 'eval_epochs': [1, 3, 6, 9, 12, 15, 18, 20], 'early_stopping': True, 'early_stop_start_epoch': 6, 'early_stop_patience': 2, 'early_stop_min_delta': 1e-05, 'negatives': 4, 'trials': [{'name': 'T8', 'dim': 8, 'hidden': [64], 'lr': 0.001, 'weight_decay': 1e-05}], 'projection_dim': 16, 'bootstrap_samples': 1000, 'min_free_gib': 5, 'publish_results': False, 'auto_resume': True, 'phases': ['validation', 'test'], 'result_datasets': {}, 'dataset_versions': {'gold': None, 'embedding': None}, 'tower_output_dim': 32}
CONFIG.update(categories=CATEGORIES_TO_RUN, variants=VARIANTS_TO_RUN, phases=PHASES,
    train_device=TRAIN_DEVICE, score_devices=SCORE_DEVICES, gold_root=GOLD_ROOT,
    embedding_root=EMBEDDING_ROOT, embedding_bundle=BUNDLE_NAME, resume_root=RESUME_ROOT,
    publish_results=PUBLISH_RESULTS, dataset_versions=INPUT_VERSIONS,
    train_batch=TRAIN_BATCH, query_batch=QUERY_BATCH, item_block=ITEM_BLOCK,
    score_shard_queries=SCORE_SHARD_QUERIES, projection_dim=PROJECTION_DIM,
    tower_output_dim=TOWER_OUTPUT_DIM, negatives=NEGATIVES_PER_POSITIVE,
    max_epochs=MAX_EPOCHS, eval_epochs=EVAL_EPOCHS, early_stopping=EARLY_STOPPING,
    early_stop_start_epoch=EARLY_STOP_START_EPOCH, early_stop_patience=EARLY_STOP_PATIENCE,
    early_stop_min_delta=EARLY_STOP_MIN_DELTA,
    trials=[dict(name='T' + str(ID_EMBED_DIM), dim=ID_EMBED_DIM, hidden=MLP_HIDDEN, lr=.001, weight_decay=.00001)],
    result_datasets={variant + '/' + category: RESULT_OWNER + '/gold-' + variant.lower()
        + '-results-' + category.lower().replace('_', '-') for variant in VARIANTS_TO_RUN for category in CATEGORIES_TO_RUN})
CONFIG_PATH = Path('/kaggle/working/two_tower_config.json')
if RUN_MODE not in ('validation', 'category_test'):
    raise ValueError("RUN_MODE phải là 'validation' hoặc 'category_test'")
if 'validation' not in PHASES and not RESUME_ROOT:
    raise ValueError('Gắn Output validation và đặt RESUME_ROOT trước khi chạy riêng phase test')
if TEST_CATEGORY not in ['All_Beauty', 'Video_Games', 'Industrial_and_Scientific']:
    raise ValueError('TEST_CATEGORY không hợp lệ')


## Huấn luyện và đánh giá

BCEWithLogitsLoss + Adam FP32, uniform negatives loại observed train items, không dùng future labels.
Các negatives là unobserved giả định, không phải dislike đã xác nhận. Sampler dùng toàn train fold
theo contract hiện có, không phải pool causal theo từng anchor. Giữ cùng sampler với NCF/DeepFM.

Validation train trước 2021-01-01, labels nửa đầu 2021; test refit từ đầu trước 2022-01-01,
labels nửa đầu 2022 (UTC). Chọn epoch bằng exact warm_full NDCG@10, rồi Recall@10,
ít tham số và epoch sớm hơn. Test dùng đúng selected_epoch, không early-stop theo test.
Suite hoàn tất validation của mọi job đã chọn trước khi chạy test.
Early stopping bắt đầu tại epoch 6, patience=2 validation checks, min_delta=0.00001.
Giữ best weights thực tế kể cả tiny gain; log phân biệt selected/stopped epoch và stop reason.

Scoring encode item một lần/checkpoint/device rồi dùng matmul theo blocks, giữ Top-20 trên device.
Quét đủ candidates, không ANN/sampled evaluation; không lọc seen items, giữ targets ngoài catalog
trong mẫu số. Unknown users dùng Popularity; known users thiếu text/history vẫn có ID prediction.
All_queries và nguyên warm_full Gold, history bins, repeat/novel, coverage và fallback được báo riêng.
CI 95% bootstrap theo query không đo biến động training giữa nhiều seeds.


In [ ]:
if not ENVIRONMENT_READY:
    raise RuntimeError('Chạy cell dependency thành công trước')
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
subprocess.run([sys.executable, '/kaggle/working/gold_neural_recommendation.py', '--config', str(CONFIG_PATH)], check=True)


## Kết quả và ablation

Output có mappings đúng fold, best_config, checkpoints, loss/validation.csv, stop state,
item_output_vectors.npy, Top-20, per-query/group metrics, CI với Popularity và resource logs.
TT2 giữ query profiles/scaler sau phase complete; prefix train cache được compact để tiết kiệm disk.
Kết quả mới là full benchmark; không so trực tiếp với pilot ALS khác cutoff/population.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '/kaggle/working')
import gold_recommendation as rec
results = json.loads((Path(CONFIG['output_root']) / 'two_tower_suite_results.json').read_text(encoding='utf-8'))
tables = []
for result in results:
    root = Path(result['run_root'])
    print(result['category'], result['variant'], root)
    display(pd.read_csv(root / 'trials.csv'))
    for path in sorted((root / 'trials').glob('*/loss.csv')):
        frame = pd.read_csv(path)
        frame.plot(x='epoch', y='loss', marker='o', title=result['category'] + ' / ' + result['variant'])
        plt.tight_layout(); plt.show()
        display(pd.read_csv(path.parent / 'validation.csv'))
    for phase in CONFIG['phases']:
        frame = pd.read_csv(root / phase / 'evaluation/metrics.csv')
        frame.insert(0, 'category', result['category']); frame.insert(1, 'variant', result['variant'])
        frame.insert(2, 'phase', phase); tables.append(frame)
metrics = pd.concat(tables, ignore_index=True)
display(metrics[metrics.cohort.isin(['all_queries', 'warm_full'])])
analysis_root = Path(CONFIG['output_root']) / 'analysis' / (CATEGORIES_TO_RUN[0] if len(CATEGORIES_TO_RUN) == 1 else 'validation-all')
analysis_root.mkdir(parents=True, exist_ok=True)
metrics.to_csv(analysis_root / 'metrics.csv', index=False)
for result in results:
    run_root = Path(result['run_root'])
    variant_dir = analysis_root / result['variant']
    variant_dir.mkdir(parents=True, exist_ok=True)
    for name in ('best_config.json', 'trials.csv'):
        source_file = run_root / name
        if source_file.is_file():
            (variant_dir / name).write_bytes(source_file.read_bytes())
    for phase in CONFIG['phases']:
        source_dir = run_root / phase / 'evaluation'
        target_dir = variant_dir / phase
        target_dir.mkdir(parents=True, exist_ok=True)
        (target_dir / 'metrics.csv').write_bytes((source_dir / 'metrics.csv').read_bytes())
with (analysis_root / 'analysis_manifest.json').open('w', encoding='utf-8') as stream:
    json.dump(dict(category=CATEGORIES_TO_RUN, variants=CONFIG['variants'], phases=CONFIG['phases'],
        dataset_versions=CONFIG.get('dataset_versions'), result_datasets=[r.get('dataset') for r in results],
        local_runs=[r['run_root'] for r in results]), stream, ensure_ascii=False, indent=2)
for category in CATEGORIES_TO_RUN:
    roots = {r['variant']: Path(r['run_root']) for r in results if r['category'] == category}
    for phase in PHASES:
        for first, second in [('TT0', 'TT1'), ('TT1', 'TT2')]:
            if first not in roots or second not in roots:
                continue
            a = roots[first] / phase / 'evaluation'
            b = roots[second] / phase / 'evaluation'
            output = Path(CONFIG['output_root']) / 'comparisons' / category / phase / (second + '-vs-' + first + '.json')
            rec.compare_results(a, b, output, samples=CONFIG['bootstrap_samples'], seed=CONFIG['seed'])
            print(category, phase, second, 'vs', first, 'system + Popularity:', json.loads(output.read_text(encoding='utf-8')))
            af = pd.read_parquet(a / (first.lower() + '_native_per_query.parquet'))
            bf = pd.read_parquet(b / (second.lower() + '_native_per_query.parquet'))
            merged = af.merge(bf, on='query_id', suffixes=('_a', '_b'), how='outer', validate='one_to_one', indicator=True)
            if not (merged['_merge'] == 'both').all() or not (merged.warm_full_a == merged.warm_full_b).all():
                raise ValueError('Native comparison populations differ')
            delta = merged['ndcg@10_b'].to_numpy() - merged['ndcg@10_a'].to_numpy()
            native_ci = {group: rec.paired_ci(delta[mask], CONFIG['bootstrap_samples'], CONFIG['seed'])
                for group, mask in {'all_queries': merged.query_id.notna().to_numpy(), 'warm_full': merged.warm_full_a.to_numpy(bool)}.items()}
            rec.save_json(output.with_name(output.stem + '-native.json'), native_ci)
            compact_comparisons = analysis_root / 'comparisons' / phase
            compact_comparisons.mkdir(parents=True, exist_ok=True)
            for comparison in (output, output.with_name(output.stem + '-native.json')):
                (compact_comparisons / comparison.name).write_bytes(comparison.read_bytes())
            print('native:', native_ci)


## Dọn run local sau khi xác minh Dataset

Cell xóa không tự chạy. Trước hết mở từng Kaggle Dataset trong `analysis_manifest.json`,
xác nhận version mới đã xử lý xong và có `result_bundle.json` cùng `recommendation_results/`.
Sau đó mới đổi cả hai cờ bên dưới thành True và chạy cell dọn. Nếu upload còn `submitted_processing`,
Dataset chưa kiểm tra được hoặc gói analysis chưa đầy đủ, để cờ False và giữ nguyên run local.
Cell chỉ xóa thư mục run của đúng category đang chọn; gói analysis nhỏ và upload receipts được giữ lại.


In [ ]:
REMOTE_DATASETS_VERIFIED = False  # xác nhận thủ công từng Dataset đã xử lý và đọc được
DELETE_LOCAL_CATEGORY_RUNS = False  # chủ động bật sau khi kiểm tra xong
if REMOTE_DATASETS_VERIFIED and DELETE_LOCAL_CATEGORY_RUNS:
    if RUN_MODE != 'category_test' or len(CATEGORIES_TO_RUN) != 1:
        raise ValueError('Dọn chỉ hỗ trợ một category ở chế độ category_test')
    category = CATEGORIES_TO_RUN[0]
    analysis_root = Path(CONFIG['output_root']) / 'analysis' / category
    required = [analysis_root / 'metrics.csv', analysis_root / 'analysis_manifest.json']
    if not all(path.is_file() for path in required):
        raise ValueError('Gói analysis category chưa hoàn tất; giữ run local')
    base = Path(CONFIG['output_root']).resolve()
    for result in results:
        root = Path(result['run_root']).resolve()
        if result['category'] != category or root.parent != base:
            raise ValueError('Run path/category không an toàn cho cleanup')
        receipt_path = base / (root.name + '_upload_receipt.json')
        receipt = rec.read_json(receipt_path)
        if receipt.get('status') != 'submitted_processing' or receipt.get('handle') != result['dataset']:
            raise ValueError('Upload receipt chưa khớp; giữ run local')
        __import__('shutil').rmtree(root)
    suite_path = base / 'two_tower_suite_results.json'
    if suite_path.is_file():
        summary = rec.read_json(suite_path)
        for row in summary:
            if row.get('category') == category:
                row['run_root'] = None
                row['analysis_root'] = str(analysis_root)
        rec.save_json(suite_path, summary)
    print('Đã dọn run local của', category, '; analysis và receipts vẫn được giữ.')
else:
    print('Cleanup đang tắt; run local được giữ nguyên.')


## Resume

Last checkpoint lưu model/optimizer/RNG và early-stop state ở ranh giới epoch; epoch dở sẽ chạy lại.
Ranking shards đã verify được dùng lại, signature khác thì từ chối. Thay kiến trúc/config/source
tạo run mới. Output working mất khi phiên mất nếu chưa Save Version/Output.
Upload chỉ xảy ra khi bật PUBLISH_RESULTS; Dataset private và remote checks như 09/10.

Tham khảo: [RecSys 2019](https://research.google/pubs/sampling-bias-corrected-neural-modeling-for-large-corpus-item-recommendations/),
[WWW 2020](https://yangli-cs-ucsb.github.io/paper/www20_yang.pdf).
Kích thước mạng, review/rating features và BCE là lựa chọn của đồ án.
